Cell 1

In [1]:
!nvidia-smi

Sun Mar 29 15:43:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Cell 2

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/trellis_project"
for folder in ["hf_cache", "outputs", "data", "models", "logs", "notebooks", "results"]:
    os.makedirs(f"{BASE}/{folder}", exist_ok=True)

print("BASE =", BASE)

Mounted at /content/drive
BASE = /content/drive/MyDrive/trellis_project


Cell 3

In [3]:
%%bash
set -e

if [ ! -d /usr/local/miniforge ]; then
  wget -qO /tmp/Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
  bash /tmp/Miniforge3.sh -b -p /usr/local/miniforge
else
  echo "Miniforge already installed"
fi

PREFIX=/usr/local/miniforge
Unpacking bootstrapper...
Unpacking payload...
Extracting ca-certificates-2026.2.25-hbd8a1cb_0.conda
Extracting libgomp-15.2.0-he0feb66_18.conda
Extracting libzlib-1.3.2-h25fd6f3_2.conda
Extracting nlohmann_json-abi-3.12.0-h0f90c79_1.conda
Extracting pybind11-abi-11-hc364b38_1.conda
Extracting python_abi-3.13-8_cp313.conda
Extracting tzdata-2025c-hc9c84f9_1.conda
Extracting _openmp_mutex-4.5-20_gnu.conda
Extracting zstd-1.5.7-hb78ec9c_6.conda
Extracting ld_impl_linux-64-2.45.1-default_hbd61a6d_101.conda
Extracting libgcc-15.2.0-he0feb66_18.conda
Extracting bzip2-1.0.8-hda65f42_9.conda
Extracting c-ares-1.34.6-hb03c661_0.conda
Extracting keyutils-1.6.3-hb9d3cd8_0.conda
Extracting libexpat-2.7.4-hecca717_0.conda
Extracting libffi-3.5.2-h3435931_0.conda
Extracting libgcc-ng-15.2.0-h69a702a_18.conda
Extracting libiconv-1.18-h3b78370_2.conda
Extracting liblzma-5.8.2-hb03c661_0.conda
Extracting libmpdec-4.0.0-hb03c661_1.conda
Extracting libstdcxx-15.2.0-h934c35e_1

Cell 4

In [4]:
import os
os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]

!conda --version
!mamba --version

conda 26.1.1
2.5.0


Cell 5

In [5]:
%%bash
set -e
cd /content

if [ ! -d /content/TRELLIS ]; then
  git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git
else
  echo "TRELLIS already present"
fi

cd /content/TRELLIS
git rev-parse HEAD

Submodule path 'trellis/representations/mesh/flexicubes': checked out '815e075a2a400d06c48d94c347674344ed6ae5c5'
442aa1e1afb9014e80681d3bf604e8d728a86ee7


Cloning into 'TRELLIS'...
Submodule 'trellis/representations/mesh/flexicubes' (https://github.com/MaxtirError/FlexiCubes.git) registered for path 'trellis/representations/mesh/flexicubes'
Cloning into '/content/TRELLIS/trellis/representations/mesh/flexicubes'...


Cell 6

In [6]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh

for i in 1 2 3; do
  if conda env list | awk '{print $1}' | grep -qx "trellis"; then
    echo "conda env 'trellis' already exists"
    break
  fi

  echo "Attempt $i: creating conda env..."
  mamba create -y -n trellis -c conda-forge python=3.10 && break

  echo "Env creation failed on attempt $i"
  sleep 10
done

conda activate trellis
python --version
which python

Attempt 1: creating conda env...


Transaction

  Prefix: /usr/local/miniforge/envs/trellis

  Updating specs:

   - python=3.10


  Package               Version  Build                 Channel           Size
───────────────────────────────────────────────────────────────────────────────
  Install:
───────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge     Cached
  + bzip2                 1.0.8  hda65f42_9            conda-forge     Cached
  + ca-certificates   2026.2.25  hbd8a1cb_0            conda-forge     Cached
  + icu                    78.3  h33c6efd_0            conda-forge     Cached
  + ld_impl_linux-64     2.45.1  default_hbd61a6d_102  conda-forge      728kB
  + libexpat              2.7.5  hecca717_0            conda-forge       77kB
  + libffi                3.5.2  h3435931_0            conda-forge     Cached
  + libgcc               15.2.0  he0feb66_18           conda-forge     Ca

Cell 7

In [7]:
import os

BASE = "/content/drive/MyDrive/trellis_project"

os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]
os.environ["HF_HOME"] = f"{BASE}/hf_cache"
os.environ["TORCH_HOME"] = f"{BASE}/hf_cache"
os.environ["SPCONV_ALGO"] = "native"
os.environ.pop("ATTN_BACKEND", None)  # leave unset for now

for k in ["HF_HOME", "TORCH_HOME", "SPCONV_ALGO", "ATTN_BACKEND"]:
    print(k, "=", os.environ.get(k))

HF_HOME = /content/drive/MyDrive/trellis_project/hf_cache
TORCH_HOME = /content/drive/MyDrive/trellis_project/hf_cache
SPCONV_ALGO = native
ATTN_BACKEND = None


Cell 8

In [8]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

python -m pip install --upgrade pip

pip uninstall -y torch torchvision torchaudio kaolin xformers || true

pip install --no-cache-dir \
  torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 \
  --index-url https://download.pytorch.org/whl/cu128

pip install --no-cache-dir \
  kaolin==0.18.0 \
  -f https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.8.0_cu128.html

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 316.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 608.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 375.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 349.9 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 344.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 335.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 345.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 341.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 341.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 347.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 332.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 

Cell 9

In [9]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

pip install --no-cache-dir easydict

Cell 10

In [10]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

pip install --no-cache-dir rembg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 292.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 265.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 257.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 265.1 MB/s  0:00:00



Cell 11

In [11]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

pip install --no-cache-dir onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 199.6 MB/s  0:00:00



Cell 12

In [12]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

pip install --no-cache-dir transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 172.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 919.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 277.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 260.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 280.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 556.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.6/790.6 kB 417.0 MB/s  0:00:00



Cell 13

In [13]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

export HF_HOME=/content/drive/MyDrive/trellis_project/hf_cache
export TORCH_HOME=/content/drive/MyDrive/trellis_project/hf_cache
export SPCONV_ALGO=native
unset ATTN_BACKEND || true

python - <<'PY'
import importlib, subprocess, sys

# keep the already-needed basics
needed = [
    ("open3d", "open3d"),
    ("plyfile", "plyfile"),
]

missing = []
for pip_name, mod_name in needed:
    try:
        importlib.import_module(mod_name)
    except Exception:
        missing.append(pip_name)

print("missing basic deps:", missing)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *missing])

# make sure we do NOT keep the unrelated tiny PyPI utils3d package
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "utils3d"], check=False)

# install the utils3d package TRELLIS expects
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "git+https://github.com/EasternJournalist/utils3d.git"
])
PY

cd /content/TRELLIS

python - <<'PY'
import torch
import kaolin
import open3d as o3d
import plyfile
import utils3d

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    x = torch.randn(1024, 1024, device="cuda")
    y = x @ x
    print("matmul ok:", y.shape)

print("kaolin: OK")
print("open3d: OK", o3d.__version__)
print("plyfile: OK", getattr(plyfile, "__version__", "imported"))
print("utils3d: OK", utils3d.__file__)
print("utils3d has torch:", hasattr(utils3d, "torch"))

from trellis.pipelines import TrellisImageTo3DPipeline
print("TRELLIS import OK")
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 316.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 314.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 332.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 359.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 453.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 331.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 334.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 228.0 MB/s  0:00:00

  Cloning https://github.com/EasternJournalist/utils3d.git to /tmp/pip-req-build-6mrlt012
  Resolved https://github.com/EasternJournalist/utils3d.git to commit 6abd5afc22c326eb8ee93945d5980dde2dcf3b3b
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: fini

  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /tmp/pip-req-build-6mrlt012


Cell 14

In [14]:
%%bash
set -e

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis
cd /content/TRELLIS

python - <<'PY'
import os, json, subprocess, torch, platform

BASE = "/content/drive/MyDrive/trellis_project"
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()

meta = {
    "trellis_commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "HF_HOME": os.environ.get("HF_HOME"),
    "TORCH_HOME": os.environ.get("TORCH_HOME"),
    "ATTN_BACKEND": os.environ.get("ATTN_BACKEND"),
    "SPCONV_ALGO": os.environ.get("SPCONV_ALGO"),
}

os.makedirs(f"{BASE}/results", exist_ok=True)
with open(f"{BASE}/results/run_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))
PY

{
  "trellis_commit": "442aa1e1afb9014e80681d3bf604e8d728a86ee7",
  "python": "3.10.20",
  "torch": "2.8.0+cu128",
  "cuda_version": "12.8",
  "cuda_available": true,
  "gpu_name": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "HF_HOME": "/content/drive/MyDrive/trellis_project/hf_cache",
  "TORCH_HOME": "/content/drive/MyDrive/trellis_project/hf_cache",
  "ATTN_BACKEND": null,
  "SPCONV_ALGO": "native"
}


Cell 14.5

In [15]:
from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/trellis_project")
REPO = Path("/content/TRELLIS")
OUT = REPO / "workflow_out_ip2p_t1"
DRIVE_OUT = BASE / "outputs" / "trellis1_ip2p_workflow"

OUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

os.environ["PATH"] = "/usr/local/miniforge/bin:" + os.environ["PATH"]
os.environ["HF_HOME"] = str(BASE / "hf_cache")
os.environ["TORCH_HOME"] = str(BASE / "hf_cache")
os.environ["SPCONV_ALGO"] = "native"

print("REPO      =", REPO)
print("OUT       =", OUT)
print("DRIVE_OUT =", DRIVE_OUT)
print("HF_HOME   =", os.environ["HF_HOME"])

REPO      = /content/TRELLIS
OUT       = /content/TRELLIS/workflow_out_ip2p_t1
DRIVE_OUT = /content/drive/MyDrive/trellis_project/outputs/trellis1_ip2p_workflow
HF_HOME   = /content/drive/MyDrive/trellis_project/hf_cache


Cell 15

In [16]:
%%bash
set -euo pipefail

unset PYTHONPATH || true
source /usr/local/miniforge/etc/profile.d/conda.sh
conda activate trellis

python - <<'PY'
import importlib, subprocess, sys

mods = [
    ("diffusers", "diffusers"),
    ("accelerate", "accelerate"),
    ("safetensors", "safetensors"),
    ("trimesh", "trimesh"),
    ("scipy", "scipy"),
]
missing = []
for pip_name, mod_name in mods:
    try:
        importlib.import_module(mod_name)
    except Exception:
        missing.append(pip_name)

print("missing:", missing)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", *missing])

import diffusers, accelerate, trimesh, scipy
print("diffusers:", diffusers.__version__)
print("accelerate:", accelerate.__version__)
print("trimesh:", trimesh.__version__)
print("scipy:", scipy.__version__)
PY

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 147.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 1.2 GB/s  0:00:00

missing: ['diffusers', 'accelerate', 'trimesh']
diffusers: 0.37.1
accelerate: 1.13.0
trimesh: 4.11.5
scipy: 1.15.3


Cell 16

In [17]:
from pathlib import Path

work_dir = Path("/content/TRELLIS/workflow_out_ip2p_t1")
work_dir.mkdir(parents=True, exist_ok=True)

# =========================
# MASTER NOTEBOOK MODE
# =========================
# "single_sample"  -> existing one-off Pix2Pix -> TRELLIS sample path
# "dataset_sweep"  -> existing bulk dataset builder path
# "adapter_train"  -> NEW: train a learned bridge from Pix2Pix-side conditioning features
#                     to TRELLIS-side input-image proxy features using the finished dataset
WORKFLOW_MODE = "adapter_train"

# =========================
# SINGLE-SAMPLE SOURCE SETTINGS
# =========================

SOURCE_MODE = "dataset"   # "upload" or "dataset"

# Single-item preview controls
DATASET_START_IDX = 1
DATASET_N_SHOW = 12
DATASET_CHOSEN_LOCAL_IDX = 0

# If SOURCE_MODE == "dataset":
DATASET_IMAGE_KIND = "edited"   # "original" or "edited"

# If SOURCE_MODE == "upload":
REUSE_EXISTING_UPLOAD = True

# =========================
# EDIT SETTINGS
# =========================

RUN_IP2P_EDIT = False

PROMPT_MODE = "dataset"   # "dataset" / "generic" / "custom"

CUSTOM_EDIT_PROMPT = "make this look like a clean single centered object on a plain background"
GENERIC_EDIT_PROMPT = "make this the same single object as a clean studio product photo, centered, plain background"
NEGATIVE_PROMPT = "extra objects, hands, people, text, clutter, reflections, busy background, cut off object"

IP2P_STEPS = 20
TEXT_GUIDANCE = 7.5
IMAGE_GUIDANCE = 1.5
IP2P_SEED = 123

# =========================
# TRELLIS SETTINGS
# =========================

T1_MODEL_ID = "microsoft/TRELLIS-image-large"
T1_SS_STEPS = 18
T1_SLAT_STEPS = 18
T1_SS_CFG = 7.5
T1_SLAT_CFG = 3.0

# =========================
# LOCAL TEACHER DATASET
# =========================

DATASET_ROOT = "/content/drive/MyDrive/trellis_project/datasets/trellis1_ip2p_teacher_pairs"
DATASET_SAMPLE_ID = ""
APPEND_DATASET_INDEX = True

SAVE_STL = True
SAVE_OBJ = True
SAVE_GLB = True
SAVE_PLY = True

# =========================
# EXISTING EXECUTION PATH FLAGS
# =========================

BUILD_DATASET_SAMPLE = (WORKFLOW_MODE == "single_sample")
RUN_DATASET_SWEEP = (WORKFLOW_MODE == "dataset_sweep")

# =========================
# EXISTING DATASET SWEEP SETTINGS
# =========================

DATASET_SWEEP_START_ID = 101
DATASET_SWEEP_END_ID = 1000
DATASET_SWEEP_MODE = "edited_direct"        # "original_only" / "edited_only" / "original_and_edit" / "edited_direct"
DATASET_SWEEP_PROMPT_MODE = "dataset"       # "dataset" / "generic" / "custom"
DATASET_SWEEP_MAX_ITEMS = None

# =========================
# NEW: ADAPTER TRAINING SETTINGS
# =========================

RUN_FEATURE_ADAPTER_TRAINING = (WORKFLOW_MODE == "adapter_train")

# Feature bridge:
#   input  = [original_image_feature || edit_instruction_text_feature]
#   target = edited_image_feature
#
# This gives a first learned bridge from Pix2Pix-side conditioning into the
# TRELLIS-input image feature space proxy using your finished teacher dataset.

ADAPTER_USE_INDEX = True
ADAPTER_SAMPLE_LIMIT = None          # None = all samples in DATASET_ROOT
ADAPTER_MIN_REQUIRED_SAMPLES = 50
ADAPTER_RANDOM_SEED = 123
ADAPTER_VAL_FRACTION = 0.10

ADAPTER_CLIP_MODEL_ID = "openai/clip-vit-large-patch14"
ADAPTER_BATCH_SIZE = 16
ADAPTER_DEVICE = "cuda"

ADAPTER_HIDDEN_DIM = 2048
ADAPTER_DROPOUT = 0.10
ADAPTER_EPOCHS = 12
ADAPTER_LR = 1e-4
ADAPTER_WEIGHT_DECAY = 1e-4

ADAPTER_COSINE_LOSS_WEIGHT = 1.0
ADAPTER_MSE_LOSS_WEIGHT = 0.25

ADAPTER_OUT_DIR = "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
ADAPTER_SAVE_EVERY_EPOCH = False

print("WORKFLOW_MODE               =", WORKFLOW_MODE)

print("SOURCE_MODE                 =", SOURCE_MODE)
print("DATASET_START_IDX           =", DATASET_START_IDX)
print("DATASET_N_SHOW              =", DATASET_N_SHOW)
print("DATASET_CHOSEN_LOCAL_IDX    =", DATASET_CHOSEN_LOCAL_IDX)
print("DATASET_IMAGE_KIND          =", DATASET_IMAGE_KIND)
print("REUSE_EXISTING_UPLOAD       =", REUSE_EXISTING_UPLOAD)

print("RUN_IP2P_EDIT               =", RUN_IP2P_EDIT)
print("PROMPT_MODE                 =", PROMPT_MODE)
print("IP2P_STEPS                  =", IP2P_STEPS)
print("TEXT_GUIDANCE               =", TEXT_GUIDANCE)
print("IMAGE_GUIDANCE              =", IMAGE_GUIDANCE)
print("IP2P_SEED                   =", IP2P_SEED)

print("T1_MODEL_ID                 =", T1_MODEL_ID)
print("T1_SS_STEPS                 =", T1_SS_STEPS)
print("T1_SLAT_STEPS               =", T1_SLAT_STEPS)
print("T1_SS_CFG                   =", T1_SS_CFG)
print("T1_SLAT_CFG                 =", T1_SLAT_CFG)

print("DATASET_ROOT                =", DATASET_ROOT)
print("BUILD_DATASET_SAMPLE        =", BUILD_DATASET_SAMPLE)
print("RUN_DATASET_SWEEP           =", RUN_DATASET_SWEEP)

print("RUN_FEATURE_ADAPTER_TRAINING=", RUN_FEATURE_ADAPTER_TRAINING)
print("ADAPTER_USE_INDEX           =", ADAPTER_USE_INDEX)
print("ADAPTER_SAMPLE_LIMIT        =", ADAPTER_SAMPLE_LIMIT)
print("ADAPTER_MIN_REQUIRED_SAMPLES=", ADAPTER_MIN_REQUIRED_SAMPLES)
print("ADAPTER_CLIP_MODEL_ID       =", ADAPTER_CLIP_MODEL_ID)
print("ADAPTER_BATCH_SIZE          =", ADAPTER_BATCH_SIZE)
print("ADAPTER_HIDDEN_DIM          =", ADAPTER_HIDDEN_DIM)
print("ADAPTER_EPOCHS              =", ADAPTER_EPOCHS)
print("ADAPTER_LR                  =", ADAPTER_LR)
print("ADAPTER_OUT_DIR             =", ADAPTER_OUT_DIR)

WORKFLOW_MODE               = adapter_train
SOURCE_MODE                 = dataset
DATASET_START_IDX           = 1
DATASET_N_SHOW              = 12
DATASET_CHOSEN_LOCAL_IDX    = 0
DATASET_IMAGE_KIND          = edited
REUSE_EXISTING_UPLOAD       = True
RUN_IP2P_EDIT               = False
PROMPT_MODE                 = dataset
IP2P_STEPS                  = 20
TEXT_GUIDANCE               = 7.5
IMAGE_GUIDANCE              = 1.5
IP2P_SEED                   = 123
T1_MODEL_ID                 = microsoft/TRELLIS-image-large
T1_SS_STEPS                 = 18
T1_SLAT_STEPS               = 18
T1_SS_CFG                   = 7.5
T1_SLAT_CFG                 = 3.0
DATASET_ROOT                = /content/drive/MyDrive/trellis_project/datasets/trellis1_ip2p_teacher_pairs
BUILD_DATASET_SAMPLE        = False
RUN_DATASET_SWEEP           = False
RUN_FEATURE_ADAPTER_TRAINING= True
ADAPTER_USE_INDEX           = True
ADAPTER_SAMPLE_LIMIT        = None
ADAPTER_MIN_REQUIRED_SAMPLES= 50
ADAPTER_CLIP_MODEL_ID       = 

Cell 16.1

In [18]:
if WORKFLOW_MODE == "adapter_train":
    print("Cell 16.1: skipped because WORKFLOW_MODE='adapter_train'")
else:
    from pathlib import Path
    import shutil
    import json
    import io
    from PIL import Image

    work_dir = Path("/content/TRELLIS/workflow_out_ip2p_t1")
    work_dir.mkdir(parents=True, exist_ok=True)

    upload_path = work_dir / "source_object.png"
    dataset_original_path = work_dir / "dataset_source.png"
    dataset_edited_path = work_dir / "dataset_edited.png"

    dataset_original_prompt_path = work_dir / "dataset_original_prompt.txt"
    dataset_edit_prompt_path = work_dir / "dataset_edit_prompt.txt"
    dataset_edited_prompt_path = work_dir / "dataset_edited_prompt.txt"

    source_choice_meta = work_dir / "source_choice.json"

    if SOURCE_MODE == "upload":
        if upload_path.exists() and REUSE_EXISTING_UPLOAD:
            print("Reusing existing upload:", upload_path)
            img = Image.open(upload_path).convert("RGBA")
        else:
            from google.colab import files
            uploaded = files.upload()
            if len(uploaded) == 0:
                raise RuntimeError("No file uploaded.")
            first_name = next(iter(uploaded))
            img = Image.open(io.BytesIO(uploaded[first_name])).convert("RGBA")
            img.save(upload_path)
            print("Saved uploaded image to:", upload_path)

        meta = {
            "source_mode": "upload",
            "upload_path": str(upload_path),
        }
        source_choice_meta.write_text(json.dumps(meta, indent=2))

        display(img)

    elif SOURCE_MODE == "dataset":
        from datasets import load_dataset
        from itertools import islice
        import matplotlib.pyplot as plt

        ds = load_dataset(
            "timbrooks/instructpix2pix-clip-filtered",
            split="train",
            streaming=True,
        )

        samples = list(islice(ds, DATASET_START_IDX, DATASET_START_IDX + DATASET_N_SHOW))

        if len(samples) == 0:
            raise RuntimeError("No dataset samples loaded for the requested preview window.")

        ncols = 4
        nrows = (len(samples) + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

        for i, ax in enumerate(axes):
            if i < len(samples):
                row = samples[i]
                ax.imshow(row["edited_image"])
                ax.set_title(
                    f"local_idx={i} | global_idx={DATASET_START_IDX + i}\nedit: {row['edit_prompt'][:55]}",
                    fontsize=9
                )
                ax.axis("off")
            else:
                ax.axis("off")

        plt.tight_layout()
        plt.show()

        chosen_idx = DATASET_CHOSEN_LOCAL_IDX
        if not (0 <= chosen_idx < len(samples)):
            raise ValueError(f"DATASET_CHOSEN_LOCAL_IDX must be between 0 and {len(samples)-1}")

        row = samples[chosen_idx]

        src_img = row["original_image"].convert("RGBA")
        edit_img = row["edited_image"].convert("RGBA")

        src_img.save(dataset_original_path)
        edit_img.save(dataset_edited_path)

        dataset_original_prompt_path.write_text(row["original_prompt"], encoding="utf-8")
        dataset_edit_prompt_path.write_text(row["edit_prompt"], encoding="utf-8")
        dataset_edited_prompt_path.write_text(row["edited_prompt"], encoding="utf-8")

        meta = {
            "source_mode": "dataset",
            "dataset_start_idx": DATASET_START_IDX,
            "dataset_n_show": DATASET_N_SHOW,
            "dataset_chosen_local_idx": chosen_idx,
            "dataset_chosen_global_idx": DATASET_START_IDX + chosen_idx,
            "dataset_image_kind": DATASET_IMAGE_KIND,
            "dataset_original_path": str(dataset_original_path),
            "dataset_edited_path": str(dataset_edited_path),
            "original_prompt": row["original_prompt"],
            "edit_prompt": row["edit_prompt"],
            "edited_prompt": row["edited_prompt"],
        }
        source_choice_meta.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

        print("Chosen dataset local_idx :", chosen_idx)
        print("Chosen dataset global_idx:", DATASET_START_IDX + chosen_idx)
        print("original_prompt          :", row["original_prompt"])
        print("edit_prompt              :", row["edit_prompt"])
        print("edited_prompt            :", row["edited_prompt"])

        display(src_img)
        display(edit_img)

    else:
        raise ValueError(f"Bad SOURCE_MODE: {SOURCE_MODE}")

Cell 16.1: skipped because WORKFLOW_MODE='adapter_train'


Cell 17

In [19]:
if WORKFLOW_MODE == "adapter_train":
    print("Cell 17: skipped because WORKFLOW_MODE='adapter_train'")
else:
    from pathlib import Path
    import json

    work_dir = Path("/content/TRELLIS/workflow_out_ip2p_t1")

    upload_path = work_dir / "source_object.png"
    dataset_original_path = work_dir / "dataset_source.png"
    dataset_edited_path = work_dir / "dataset_edited.png"

    dataset_original_prompt_path = work_dir / "dataset_original_prompt.txt"
    dataset_edit_prompt_path = work_dir / "dataset_edit_prompt.txt"
    dataset_edited_prompt_path = work_dir / "dataset_edited_prompt.txt"

    if SOURCE_MODE == "upload":
        SELECTED_SOURCE_PATH = upload_path
        SELECTED_SOURCE_KIND = "upload"
        ORIGINAL_TEXT = None
        DATASET_EDIT_PROMPT = None
        EDITED_TEXT = None

    elif SOURCE_MODE == "dataset":
        if DATASET_IMAGE_KIND == "original":
            SELECTED_SOURCE_PATH = dataset_original_path
            SELECTED_SOURCE_KIND = "dataset_original"
        elif DATASET_IMAGE_KIND == "edited":
            SELECTED_SOURCE_PATH = dataset_edited_path
            SELECTED_SOURCE_KIND = "dataset_edited"
        else:
            raise ValueError(f"Bad DATASET_IMAGE_KIND: {DATASET_IMAGE_KIND}")

        ORIGINAL_TEXT = dataset_original_prompt_path.read_text(encoding="utf-8").strip() if dataset_original_prompt_path.exists() else None
        DATASET_EDIT_PROMPT = dataset_edit_prompt_path.read_text(encoding="utf-8").strip() if dataset_edit_prompt_path.exists() else None
        EDITED_TEXT = dataset_edited_prompt_path.read_text(encoding="utf-8").strip() if dataset_edited_prompt_path.exists() else None

    else:
        raise ValueError(f"Bad SOURCE_MODE: {SOURCE_MODE}")

    if not SELECTED_SOURCE_PATH.exists():
        raise FileNotFoundError(f"Missing selected source: {SELECTED_SOURCE_PATH}")

    if RUN_IP2P_EDIT:
        if PROMPT_MODE == "dataset":
            if DATASET_EDIT_PROMPT is None:
                raise RuntimeError("PROMPT_MODE='dataset' but no dataset edit prompt exists.")
            RESOLVED_EDIT_PROMPT = DATASET_EDIT_PROMPT
        elif PROMPT_MODE == "generic":
            RESOLVED_EDIT_PROMPT = GENERIC_EDIT_PROMPT
        elif PROMPT_MODE == "custom":
            RESOLVED_EDIT_PROMPT = CUSTOM_EDIT_PROMPT
        else:
            raise ValueError(f"Bad PROMPT_MODE: {PROMPT_MODE}")
    else:
        RESOLVED_EDIT_PROMPT = None

    resolved_meta = {
        "selected_source_path": str(SELECTED_SOURCE_PATH),
        "selected_source_kind": SELECTED_SOURCE_KIND,
        "original_text": ORIGINAL_TEXT,
        "dataset_edit_prompt": DATASET_EDIT_PROMPT,
        "edited_text": EDITED_TEXT,
        "run_ip2p_edit": bool(RUN_IP2P_EDIT),
        "resolved_edit_prompt": RESOLVED_EDIT_PROMPT,
    }
    (work_dir / "resolved_choice.json").write_text(json.dumps(resolved_meta, indent=2, ensure_ascii=False), encoding="utf-8")

    print("SELECTED_SOURCE_PATH  =", SELECTED_SOURCE_PATH)
    print("SELECTED_SOURCE_KIND  =", SELECTED_SOURCE_KIND)
    print("RUN_IP2P_EDIT         =", RUN_IP2P_EDIT)
    print("RESOLVED_EDIT_PROMPT  =", RESOLVED_EDIT_PROMPT)
    print("ORIGINAL_TEXT         =", ORIGINAL_TEXT)
    print("EDITED_TEXT           =", EDITED_TEXT)
    print("T1_MODEL_ID           =", T1_MODEL_ID)

Cell 17: skipped because WORKFLOW_MODE='adapter_train'


Cell 18

In [20]:
if WORKFLOW_MODE == "adapter_train":
    print("Cell 18: skipped because WORKFLOW_MODE='adapter_train'")
else:
    import os
    import subprocess
    import textwrap
    import json
    from pathlib import Path
    from PIL import Image

    work_dir = Path("/content/TRELLIS/workflow_out_ip2p_t1")
    work_dir.mkdir(parents=True, exist_ok=True)

    selected_source_path = Path(SELECTED_SOURCE_PATH)

    final_rgb_path = work_dir / "final_input_rgb.png"
    final_rgba_path = work_dir / "final_input_rgba.png"
    final_preview_path = work_dir / "final_input_preview.png"
    workflow_json_path = work_dir / "workflow_choice.json"

    if not RUN_IP2P_EDIT:
        src_rgb = Image.open(selected_source_path).convert("RGB")
        src_rgba = Image.open(selected_source_path).convert("RGBA")

        src_rgb.save(final_rgb_path)
        src_rgba.save(final_rgba_path)
        src_rgb.save(final_preview_path)

        meta = {
            "selected_source_path": str(selected_source_path),
            "run_ip2p_edit": False,
            "resolved_edit_prompt": None,
            "final_rgb_path": str(final_rgb_path),
            "final_rgba_path": str(final_rgba_path),
        }
        workflow_json_path.write_text(json.dumps(meta, indent=2))

        print("No edit requested.")
        print("WROTE", final_rgb_path)
        print("WROTE", final_rgba_path)
        print("WROTE", final_preview_path)
        display(src_rgb)

    else:
        repo_root = "/content/TRELLIS"
        env_python = "/usr/local/miniforge/envs/trellis/bin/python"
        script_path = Path("/content/TRELLIS/run_ip2p_edit_for_trellis1.py")

        script_path.write_text(textwrap.dedent(f"""
import io
import json
import torch
from pathlib import Path
from PIL import Image

from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler
from rembg import remove

src_path = Path({str(selected_source_path)!r})
out_dir = Path({str(work_dir)!r})
out_dir.mkdir(parents=True, exist_ok=True)

if not src_path.exists():
    raise FileNotFoundError(src_path)

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    "timbrooks/instruct-pix2pix",
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")

image = Image.open(src_path).convert("RGB")

edited = pipe(
    prompt={RESOLVED_EDIT_PROMPT!r},
    image=image,
    negative_prompt={NEGATIVE_PROMPT!r},
    num_inference_steps={int(IP2P_STEPS)},
    image_guidance_scale={float(IMAGE_GUIDANCE)},
    guidance_scale={float(TEXT_GUIDANCE)},
    generator=torch.Generator(device="cuda").manual_seed({int(IP2P_SEED)}),
).images[0]

rgb_path = out_dir / "pix2pix_edited_rgb.png"
rgba_path = out_dir / "pix2pix_edited_rgba.png"
preview_path = out_dir / "pix2pix_preview.png"

edited.save(rgb_path)
edited.save(preview_path)

rgba = remove(edited)
if isinstance(rgba, bytes):
    rgba = Image.open(io.BytesIO(rgba)).convert("RGBA")
else:
    rgba = rgba.convert("RGBA")
rgba.save(rgba_path)

meta = {{
    "selected_source_path": str(src_path),
    "resolved_edit_prompt": {RESOLVED_EDIT_PROMPT!r},
    "negative_prompt": {NEGATIVE_PROMPT!r},
    "steps": {int(IP2P_STEPS)},
    "image_guidance_scale": {float(IMAGE_GUIDANCE)},
    "guidance_scale": {float(TEXT_GUIDANCE)},
    "seed": {int(IP2P_SEED)},
    "rgb_path": str(rgb_path),
    "rgba_path": str(rgba_path),
    "preview_path": str(preview_path),
}}
(out_dir / "pix2pix_run_meta.json").write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
"""))

        env = os.environ.copy()
        env["HF_HOME"] = "/content/drive/MyDrive/trellis_project/hf_cache"
        env["TORCH_HOME"] = "/content/drive/MyDrive/trellis_project/hf_cache"

        run = subprocess.run(
            [env_python, str(script_path)],
            env=env,
            text=True,
            capture_output=True,
        )

        print("=== PIX2PIX STDOUT ===")
        print(run.stdout)
        print("=== PIX2PIX STDERR ===")
        print(run.stderr)

        if run.returncode != 0:
            raise RuntimeError(f"Pix2Pix edit failed with exit code {run.returncode}")

        edited_rgb_path = work_dir / "pix2pix_edited_rgb.png"
        edited_rgba_path = work_dir / "pix2pix_edited_rgba.png"
        preview_path = work_dir / "pix2pix_preview.png"

        if not edited_rgb_path.exists():
            raise FileNotFoundError(edited_rgb_path)
        if not edited_rgba_path.exists():
            raise FileNotFoundError(edited_rgba_path)

        Image.open(edited_rgb_path).convert("RGB").save(final_rgb_path)
        Image.open(edited_rgba_path).convert("RGBA").save(final_rgba_path)
        Image.open(preview_path if preview_path.exists() else edited_rgb_path).convert("RGB").save(final_preview_path)

        meta = {
            "selected_source_path": str(selected_source_path),
            "run_ip2p_edit": True,
            "resolved_edit_prompt": RESOLVED_EDIT_PROMPT,
            "negative_prompt": NEGATIVE_PROMPT,
            "final_rgb_path": str(final_rgb_path),
            "final_rgba_path": str(final_rgba_path),
            "pix2pix_rgb_path": str(edited_rgb_path),
            "pix2pix_rgba_path": str(edited_rgba_path),
        }
        workflow_json_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")

        print("Edit complete.")
        print("WROTE", final_rgb_path)
        print("WROTE", final_rgba_path)
        print("WROTE", final_preview_path)
        display(Image.open(final_preview_path))

Cell 18: skipped because WORKFLOW_MODE='adapter_train'


Cell 19

In [21]:
if WORKFLOW_MODE == "adapter_train":
    print("Cell 19: skipped because WORKFLOW_MODE='adapter_train'")
else:
    import os
    import json
    import sys
    import subprocess
    import textwrap
    from pathlib import Path

    work_dir = Path("/content/TRELLIS/workflow_out_ip2p_t1")
    work_dir.mkdir(parents=True, exist_ok=True)

    input_rgba_path = work_dir / "final_input_rgba.png"
    if not input_rgba_path.exists():
        raise FileNotFoundError(f"Cell 19 expected input image at: {input_rgba_path}")

    env_python = "/usr/local/miniforge/envs/trellis/bin/python"
    repo_root = Path("/content/TRELLIS")

    env = os.environ.copy()
    env["HF_HOME"] = "/content/drive/MyDrive/trellis_project/hf_cache"
    env["TORCH_HOME"] = "/content/drive/MyDrive/trellis_project/hf_cache"
    env["PYTHONPATH"] = f"{repo_root}:{env.get('PYTHONPATH', '')}"
    env["PATH"] = "/usr/local/cuda/bin:/usr/bin:/bin:" + env.get("PATH", "")
    env["CC"] = "/usr/bin/gcc"
    env["CXX"] = "/usr/bin/g++"
    env["CUDA_HOME"] = "/usr/local/cuda"
    env["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + env.get("LD_LIBRARY_PATH", "")
    env["SPCONV_ALGO"] = "native"
    env["ATTN_BACKEND"] = "sdpa"
    env["SPARSE_ATTN_BACKEND"] = "flash_attn"
    env["MAX_JOBS"] = "2"
    env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

    print("=== Cell 19 INPUT CHECK ===")
    print("input_rgba_path =", input_rgba_path)
    print("repo_root       =", repo_root)
    print("env_python      =", env_python)
    print("HF_HOME         =", env["HF_HOME"])
    print("CUDA_HOME       =", env["CUDA_HOME"])

    check_spconv = subprocess.run(
        [env_python, "-c", "import spconv.pytorch as spconv; print(spconv.__file__)"],
        env=env,
        text=True,
        capture_output=True,
    )
    print("=== SPCONV CHECK STDOUT ===")
    print(check_spconv.stdout)
    print("=== SPCONV CHECK STDERR ===")
    print(check_spconv.stderr)
    print("=== SPCONV CHECK EXIT CODE ===", check_spconv.returncode)
    if check_spconv.returncode != 0:
        raise RuntimeError("Cell 19 failed: spconv is not importable in the trellis env.")

    flash_stub_path = repo_root / "flash_attn.py"
    flash_stub_path.write_text(textwrap.dedent("""
import torch
import torch.nn.functional as F

def _prepare_qkv(q, k, v):
    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:
        raise ValueError(f"Expected q,k,v to have shape [B, N, H, D]; got {q.shape}, {k.shape}, {v.shape}")
    q2 = q.permute(0, 2, 1, 3).contiguous()
    k2 = k.permute(0, 2, 1, 3).contiguous()
    v2 = v.permute(0, 2, 1, 3).contiguous()
    return q2, k2, v2

def _restore(x):
    return x.permute(0, 2, 1, 3).contiguous()

def flash_attn_qkvpacked_func(qkv, dropout_p=0.0, softmax_scale=None, causal=False):
    if qkv.ndim != 5 or qkv.shape[3] != 3:
        raise ValueError(f"Expected qkv packed shape [B, N, H, 3, D], got {qkv.shape}")
    q = qkv[:, :, :, 0, :]
    k = qkv[:, :, :, 1, :]
    v = qkv[:, :, :, 2, :]
    q2, k2, v2 = _prepare_qkv(q, k, v)
    out = F.scaled_dot_product_attention(q2, k2, v2, attn_mask=None, dropout_p=0.0, is_causal=causal)
    return _restore(out)

def flash_attn_func(q, k, v, dropout_p=0.0, softmax_scale=None, causal=False):
    q2, k2, v2 = _prepare_qkv(q, k, v)
    out = F.scaled_dot_product_attention(q2, k2, v2, attn_mask=None, dropout_p=0.0, is_causal=causal)
    return _restore(out)
"""))
    print("WROTE", flash_stub_path)

    sparse_dir = repo_root / "trellis" / "modules" / "sparse"
    (sparse_dir / "attention_full.py").write_text(textwrap.dedent("""
import torch
from flash_attn import flash_attn_qkvpacked_func

def sparse_scaled_dot_product_attention(qkv, layout_crow_indices, layout_col_indices, layout_values, block_size):
    if qkv.ndim != 5:
        raise ValueError(f"Expected qkv shape [B, N, H, 3, D], got {qkv.shape}")
    return flash_attn_qkvpacked_func(qkv, dropout_p=0.0, causal=False)
"""))

    (sparse_dir / "attention_serialized.py").write_text(textwrap.dedent("""
import torch
from flash_attn import flash_attn_qkvpacked_func

def sparse_scaled_dot_product_attention_serialized(qkv, layout_crow_indices, layout_col_indices, layout_values, block_size):
    if qkv.ndim != 5:
        raise ValueError(f"Expected qkv shape [B, N, H, 3, D], got {qkv.shape}")
    return flash_attn_qkvpacked_func(qkv, dropout_p=0.0, causal=False)
"""))

    (sparse_dir / "attention_windowed.py").write_text(textwrap.dedent("""
import torch
from flash_attn import flash_attn_qkvpacked_func

def sparse_scaled_dot_product_attention_windowed(qkv, layout_crow_indices, layout_col_indices, layout_values, block_size, window_size):
    if qkv.ndim != 5:
        raise ValueError(f"Expected qkv shape [B, N, H, 3, D], got {qkv.shape}")
    return flash_attn_qkvpacked_func(qkv, dropout_p=0.0, causal=False)
"""))

    print("Patched sparse attention shims in", sparse_dir)

    pipeline_init = repo_root / "trellis" / "pipelines" / "__init__.py"
    pipeline_init.write_text("from .trellis_image_to_3d import TrellisImageTo3DPipeline\n")
    print("Patched", pipeline_init)

    script_path = repo_root / "run_trellis1_export.py"
    script_path.write_text(textwrap.dedent(f"""
import os
import sys
import json
from pathlib import Path

import trimesh
from PIL import Image
from huggingface_hub import snapshot_download

repo_root = Path("/content/TRELLIS")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

os.environ["SPCONV_ALGO"] = "native"
os.environ["ATTN_BACKEND"] = "sdpa"
os.environ["SPARSE_ATTN_BACKEND"] = "flash_attn"

input_image_path = Path({str(input_rgba_path)!r})
output_dir = Path({str(work_dir)!r})
output_dir.mkdir(parents=True, exist_ok=True)

if not input_image_path.exists():
    raise FileNotFoundError(input_image_path)

from trellis.pipelines import TrellisImageTo3DPipeline

model_snapshot_file = output_dir / "_model_snapshot_path.txt"
if model_snapshot_file.exists():
    model_dir = Path(model_snapshot_file.read_text().strip())
else:
    model_dir = Path(snapshot_download(
        repo_id={T1_MODEL_ID!r},
        cache_dir=os.environ["HF_HOME"],
    ))
    model_snapshot_file.write_text(str(model_dir))

image = Image.open(input_image_path).convert("RGBA")

pipeline = TrellisImageTo3DPipeline.from_pretrained(str(model_dir))
pipeline.cuda()

outputs = pipeline.run(
    image,
    seed={IP2P_SEED},
    sparse_structure_sampler_params={{
        "steps": {T1_SS_STEPS},
        "cfg_strength": {T1_SS_CFG},
    }},
    slat_sampler_params={{
        "steps": {T1_SLAT_STEPS},
        "cfg_strength": {T1_SLAT_CFG},
    }},
)

mesh = outputs["mesh"][0]
verts = mesh.vertices.detach().float().cpu().numpy()
faces = mesh.faces.detach().int().cpu().numpy()

raw_mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
raw_mesh.remove_unreferenced_vertices()

saved = {{}}

glb_path = output_dir / "trellis1.glb"
raw_obj_path = output_dir / "trellis1_raw_mesh.obj"
raw_stl_path = output_dir / "trellis1_raw_mesh.stl"
gaussian_ply_path = output_dir / "trellis1_gaussian.ply"
meta_path = output_dir / "trellis1_meta.json"

raw_mesh.export(glb_path)
raw_mesh.export(raw_obj_path)
raw_mesh.export(raw_stl_path)
outputs["gaussian"][0].save_ply(gaussian_ply_path)

saved["glb"] = str(glb_path.name)
saved["raw_obj"] = str(raw_obj_path.name)
saved["raw_stl"] = str(raw_stl_path.name)
saved["gaussian_ply"] = str(gaussian_ply_path.name)

meta = {{
    "input_image_path": str(input_image_path),
    "model_id": {T1_MODEL_ID!r},
    "seed": {IP2P_SEED},
    "ss_steps": {T1_SS_STEPS},
    "ss_cfg": {T1_SS_CFG},
    "slat_steps": {T1_SLAT_STEPS},
    "slat_cfg": {T1_SLAT_CFG},
    "saved": saved,
    "mesh_vertices": int(raw_mesh.vertices.shape[0]),
    "mesh_faces": int(raw_mesh.faces.shape[0]),
}}
meta_path.write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
"""))
    print("WROTE", script_path)

    run = subprocess.run(
        [env_python, str(script_path)],
        env=env,
        text=True,
        capture_output=True,
    )

    print("=== TRELLIS STDOUT ===")
    print(run.stdout)
    print("=== TRELLIS STDERR ===")
    print(run.stderr)
    print("=== TRELLIS EXIT CODE ===", run.returncode)

    expected = [
        work_dir / "trellis1.glb",
        work_dir / "trellis1_raw_mesh.obj",
        work_dir / "trellis1_raw_mesh.stl",
        work_dir / "trellis1_gaussian.ply",
        work_dir / "trellis1_meta.json",
    ]

    for p in expected:
        print("EXISTS", p, "->", p.exists())

    if run.returncode != 0:
        raise RuntimeError("Cell 19 failed. Paste only this cell's new stdout/stderr.")

Cell 19: skipped because WORKFLOW_MODE='adapter_train'


Cell 20

In [23]:
from pathlib import Path
from datetime import datetime
import json
import shutil

work_dir = Path("/content/TRELLIS/workflow_out_ip2p_t1")
dataset_root = Path(DATASET_ROOT)
dataset_root.mkdir(parents=True, exist_ok=True)

if not BUILD_DATASET_SAMPLE:
    print("Skipping dataset sample save because BUILD_DATASET_SAMPLE=False")

else:
    final_input_path = work_dir / "final_input_rgba.png"
    if not final_input_path.exists():
        raise FileNotFoundError(f"Missing final TRELLIS input image: {final_input_path}")

    trellis_meta_path = work_dir / "trellis1_meta.json"
    if not trellis_meta_path.exists():
        raise FileNotFoundError(f"Missing TRELLIS output metadata: {trellis_meta_path}")

    original_2d_name = "original_2d.png"
    original_text_name = "original_text.txt"
    edit_instruction_name = "edit_instruction.txt"
    edited_2d_name = "edited_2d.png"
    edited_text_name = "edited_text.txt"

    sample_id = datetime.now().strftime("sample_%Y%m%d_%H%M%S")
    sample_dir = dataset_root / sample_id
    sample_dir.mkdir(parents=True, exist_ok=False)

    if SOURCE_MODE == "upload":
        source_upload = work_dir / "source_object.png"
        if not source_upload.exists():
            raise FileNotFoundError(f"Missing upload source image: {source_upload}")

        shutil.copy2(source_upload, sample_dir / original_2d_name)
        (sample_dir / original_text_name).write_text("", encoding="utf-8")

        if RUN_IP2P_EDIT:
            (sample_dir / edit_instruction_name).write_text(
                RESOLVED_EDIT_PROMPT if RESOLVED_EDIT_PROMPT is not None else "",
                encoding="utf-8"
            )
            shutil.copy2(final_input_path, sample_dir / edited_2d_name)
            (sample_dir / edited_text_name).write_text("", encoding="utf-8")
        else:
            (sample_dir / edit_instruction_name).write_text("", encoding="utf-8")
            shutil.copy2(source_upload, sample_dir / edited_2d_name)
            (sample_dir / edited_text_name).write_text("", encoding="utf-8")

    elif SOURCE_MODE == "dataset":
        dataset_original_path = work_dir / "dataset_source.png"
        dataset_edited_path = work_dir / "dataset_edited.png"
        dataset_original_prompt_path = work_dir / "dataset_original_prompt.txt"
        dataset_edit_prompt_path = work_dir / "dataset_edit_prompt.txt"
        dataset_edited_prompt_path = work_dir / "dataset_edited_prompt.txt"

        if not dataset_original_path.exists():
            raise FileNotFoundError(f"Missing dataset original image: {dataset_original_path}")
        if not dataset_edited_path.exists():
            raise FileNotFoundError(f"Missing dataset edited image: {dataset_edited_path}")

        original_text = dataset_original_prompt_path.read_text(encoding="utf-8").strip() if dataset_original_prompt_path.exists() else ""
        dataset_edit_text = dataset_edit_prompt_path.read_text(encoding="utf-8").strip() if dataset_edit_prompt_path.exists() else ""
        edited_text = dataset_edited_prompt_path.read_text(encoding="utf-8").strip() if dataset_edited_prompt_path.exists() else ""

        shutil.copy2(dataset_original_path, sample_dir / original_2d_name)
        (sample_dir / original_text_name).write_text(original_text, encoding="utf-8")

        if RUN_IP2P_EDIT:
            (sample_dir / edit_instruction_name).write_text(
                RESOLVED_EDIT_PROMPT if RESOLVED_EDIT_PROMPT is not None else "",
                encoding="utf-8"
            )
            shutil.copy2(final_input_path, sample_dir / edited_2d_name)
            (sample_dir / edited_text_name).write_text(edited_text, encoding="utf-8")
        else:
            (sample_dir / edit_instruction_name).write_text(dataset_edit_text, encoding="utf-8")
            shutil.copy2(dataset_edited_path, sample_dir / edited_2d_name)
            (sample_dir / edited_text_name).write_text(edited_text, encoding="utf-8")

    else:
        raise ValueError(f"Bad SOURCE_MODE: {SOURCE_MODE}")

    if SAVE_GLB and (work_dir / "trellis1.glb").exists():
        shutil.copy2(work_dir / "trellis1.glb", sample_dir / "trellis1.glb")

    if SAVE_OBJ and (work_dir / "trellis1_raw_mesh.obj").exists():
        shutil.copy2(work_dir / "trellis1_raw_mesh.obj", sample_dir / "trellis1_raw_mesh.obj")

    if SAVE_STL and (work_dir / "trellis1_raw_mesh.stl").exists():
        shutil.copy2(work_dir / "trellis1_raw_mesh.stl", sample_dir / "trellis1_raw_mesh.stl")

    if SAVE_PLY and (work_dir / "trellis1_gaussian.ply").exists():
        shutil.copy2(work_dir / "trellis1_gaussian.ply", sample_dir / "trellis1_gaussian.ply")

    shutil.copy2(trellis_meta_path, sample_dir / "trellis1_meta.json")

    record = {
        "sample_id": sample_id,
        "source_mode": SOURCE_MODE,
        "dataset_image_kind": DATASET_IMAGE_KIND if SOURCE_MODE == "dataset" else None,
        "run_ip2p_edit": bool(RUN_IP2P_EDIT),
        "prompt_mode": PROMPT_MODE if RUN_IP2P_EDIT else None,
        "resolved_edit_prompt": RESOLVED_EDIT_PROMPT if RUN_IP2P_EDIT else None,
        "original_2d": original_2d_name,
        "original_text": original_text_name,
        "edit_instruction": edit_instruction_name,
        "edited_2d": edited_2d_name,
        "edited_text": edited_text_name,
        "trellis_glb": "trellis1.glb" if (sample_dir / "trellis1.glb").exists() else None,
        "trellis_raw_obj": "trellis1_raw_mesh.obj" if (sample_dir / "trellis1_raw_mesh.obj").exists() else None,
        "trellis_raw_stl": "trellis1_raw_mesh.stl" if (sample_dir / "trellis1_raw_mesh.stl").exists() else None,
        "trellis_gaussian_ply": "trellis1_gaussian.ply" if (sample_dir / "trellis1_gaussian.ply").exists() else None,
        "trellis_meta": "trellis1_meta.json",
        "created_at": datetime.now().isoformat(),
    }

    (sample_dir / "record.json").write_text(
        json.dumps(record, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    print("Saved dataset sample to:", sample_dir)
    print(json.dumps(record, indent=2, ensure_ascii=False))

Skipping dataset sample save because BUILD_DATASET_SAMPLE=False


Cell 20.1

In [25]:
# Cell 20.1 — append dataset metadata to index.jsonl

from pathlib import Path
import json
from datetime import datetime

DATASET_ROOT = Path(
    globals().get(
        "DATASET_ROOT",
        "/content/drive/MyDrive/trellis_project/datasets/trellis1_ip2p_teacher_pairs"
    )
)
INDEX_PATH = DATASET_ROOT / "index.jsonl"

if not BUILD_DATASET_SAMPLE:
    print("Skipping dataset sample save because BUILD_DATASET_SAMPLE=False")
else:
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)

    # Prefer the sample_dir created in Cell 20 if it exists.
    sample_dir = globals().get("sample_dir", None)

    # Fallback: find the most recent dataset sample folder that has the expected files.
    if sample_dir is None:
        candidates = []
        for p in DATASET_ROOT.iterdir():
            if p.is_dir() and (p / "original_2d.png").exists() and (p / "edited_2d.png").exists():
                candidates.append(p)
        if not candidates:
            raise FileNotFoundError(
                f"No dataset sample folder found under {DATASET_ROOT} "
                f"and sample_dir was not defined by Cell 20."
            )
        sample_dir = max(candidates, key=lambda p: p.stat().st_mtime)

    sample_dir = Path(sample_dir)

    def rel_or_none(p: Path):
        return str(p.relative_to(DATASET_ROOT)) if p.exists() else None

    original_2d = sample_dir / "original_2d.png"
    edited_2d = sample_dir / "edited_2d.png"
    edit_instruction_txt = sample_dir / "edit_instruction.txt"

    trellis_glb = sample_dir / "trellis1.glb"
    trellis_obj = sample_dir / "trellis1_raw_mesh.obj"
    trellis_stl = sample_dir / "trellis1_raw_mesh.stl"
    trellis_ply = sample_dir / "trellis1_gaussian.ply"
    trellis_meta = sample_dir / "trellis1_meta.json"

    if not original_2d.exists():
        raise FileNotFoundError(f"Missing file: {original_2d}")
    if not edited_2d.exists():
        raise FileNotFoundError(f"Missing file: {edited_2d}")
    if not edit_instruction_txt.exists():
        raise FileNotFoundError(f"Missing file: {edit_instruction_txt}")

    edit_instruction = edit_instruction_txt.read_text(encoding="utf-8").strip()

    record = {
        "sample_id": sample_dir.name,
        "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "paths": {
            "original_2d": rel_or_none(original_2d),
            "edited_2d": rel_or_none(edited_2d),
            "edit_instruction_txt": rel_or_none(edit_instruction_txt),
            "trellis_glb": rel_or_none(trellis_glb),
            "trellis_obj": rel_or_none(trellis_obj),
            "trellis_stl": rel_or_none(trellis_stl),
            "trellis_gaussian_ply": rel_or_none(trellis_ply),
            "trellis_meta_json": rel_or_none(trellis_meta),
        },
        "edit_instruction": edit_instruction,
    }

    existing_ids = set()
    if INDEX_PATH.exists():
        with open(INDEX_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    existing_ids.add(json.loads(line).get("sample_id"))
                except Exception:
                    pass

    if record["sample_id"] in existing_ids:
        print(f"index.jsonl already contains sample_id={record['sample_id']}; skipping append.")
    else:
        with open(INDEX_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Appended dataset record to {INDEX_PATH}")
        print(json.dumps(record, indent=2, ensure_ascii=False))

Skipping dataset sample save because BUILD_DATASET_SAMPLE=False


Cell 21

In [27]:
# Cell 21 — build supervised CLIP features for adapter training

from pathlib import Path
import json
from PIL import Image
from tqdm.auto import tqdm
import torch
from transformers import CLIPModel, CLIPProcessor

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 21: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Cell 21 device:", device)

    DATASET_ROOT = Path(
        globals().get(
            "DATASET_ROOT",
            "/content/drive/MyDrive/trellis_project/datasets/trellis1_ip2p_teacher_pairs"
        )
    )
    MODEL_OUTPUT_DIR = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    CLIP_MODEL_ID = globals().get("CLIP_MODEL_ID", "openai/clip-vit-large-patch14")
    BATCH_SIZE = int(globals().get("FEATURE_BATCH_SIZE", 16))

    INDEX_PATH = DATASET_ROOT / "index.jsonl"
    FEATURE_CACHE_PATH = MODEL_OUTPUT_DIR / "supervised_clip_features.pt"
    FEATURE_CACHE_PATH_ALT1 = MODEL_OUTPUT_DIR / "bridge_features.pt"
    FEATURE_CACHE_PATH_ALT2 = MODEL_OUTPUT_DIR / "adapter_features.pt"
    FEATURE_META_PATH = MODEL_OUTPUT_DIR / "supervised_clip_features_meta.json"

    def _resolve_sample_from_record(record: dict):
        paths = record.get("paths", {}) if isinstance(record.get("paths", {}), dict) else {}

        original_rel = paths.get("original_2d") or "original_2d.png"
        edited_rel = paths.get("edited_2d") or "edited_2d.png"
        instr_rel = paths.get("edit_instruction_txt") or "edit_instruction.txt"

        sample_id = record.get("sample_id", "")
        sample_dir = DATASET_ROOT / sample_id if sample_id else None

        def _resolve(rel_path, fallback_name):
            rel_path = rel_path or fallback_name
            p = DATASET_ROOT / rel_path
            if p.exists():
                return p
            if sample_dir is not None:
                p2 = sample_dir / fallback_name
                if p2.exists():
                    return p2
            return None

        original_path = _resolve(original_rel, "original_2d.png")
        edited_path = _resolve(edited_rel, "edited_2d.png")
        instr_path = _resolve(instr_rel, "edit_instruction.txt")

        if original_path is None or edited_path is None or instr_path is None:
            return None

        edit_instruction = (record.get("edit_instruction") or "").strip()
        if not edit_instruction:
            edit_instruction = instr_path.read_text(encoding="utf-8").strip()

        return {
            "sample_id": sample_id or original_path.parent.name,
            "original_path": str(original_path),
            "edited_path": str(edited_path),
            "instruction_path": str(instr_path),
            "edit_instruction": edit_instruction,
        }

    samples = []

    if INDEX_PATH.exists():
        with open(INDEX_PATH, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    record = json.loads(line)
                    sample = _resolve_sample_from_record(record)
                    if sample is not None:
                        samples.append(sample)
                except Exception:
                    continue
    else:
        for p in sorted(DATASET_ROOT.iterdir()):
            if not p.is_dir():
                continue
            original_path = p / "original_2d.png"
            edited_path = p / "edited_2d.png"
            instr_path = p / "edit_instruction.txt"
            if original_path.exists() and edited_path.exists() and instr_path.exists():
                samples.append({
                    "sample_id": p.name,
                    "original_path": str(original_path),
                    "edited_path": str(edited_path),
                    "instruction_path": str(instr_path),
                    "edit_instruction": instr_path.read_text(encoding="utf-8").strip(),
                })

    if len(samples) == 0:
        raise FileNotFoundError(f"No usable dataset samples found under {DATASET_ROOT}")

    print("Cell 21 samples:", len(samples))

    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, use_fast=False)
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
    clip_model.eval()

    def l2_normalize(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
        return x / x.norm(dim=-1, keepdim=True).clamp(min=eps)

    X_chunks = []
    Y_chunks = []
    sample_ids = []
    original_paths = []
    edited_paths = []
    instructions = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(samples), BATCH_SIZE), desc="Extracting bridge features"):
            batch = samples[start:start + BATCH_SIZE]

            original_images = [
                Image.open(item["original_path"]).convert("RGB")
                for item in batch
            ]
            edited_images = [
                Image.open(item["edited_path"]).convert("RGB")
                for item in batch
            ]
            edit_texts = [item["edit_instruction"] for item in batch]

            original_inputs = processor(
                images=original_images,
                return_tensors="pt"
            )
            edited_inputs = processor(
                images=edited_images,
                return_tensors="pt"
            )
            text_inputs = processor(
                text=edit_texts,
                padding=True,
                truncation=True,
                return_tensors="pt"
            )

            pixel_values_original = original_inputs["pixel_values"].to(device, non_blocking=True)
            pixel_values_edited = edited_inputs["pixel_values"].to(device, non_blocking=True)
            input_ids = text_inputs["input_ids"].to(device, non_blocking=True)
            attention_mask = text_inputs["attention_mask"].to(device, non_blocking=True)

            # Use CLIP submodules directly so this stays robust even if get_*_features
            # returns a model-output object in this runtime.
            original_vision_outputs = clip_model.vision_model(
                pixel_values=pixel_values_original,
                return_dict=True
            )
            edited_vision_outputs = clip_model.vision_model(
                pixel_values=pixel_values_edited,
                return_dict=True
            )
            text_outputs = clip_model.text_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )

            original_image_features = clip_model.visual_projection(
                original_vision_outputs.pooler_output
            ).float()
            edited_image_features = clip_model.visual_projection(
                edited_vision_outputs.pooler_output
            ).float()
            edit_text_features = clip_model.text_projection(
                text_outputs.pooler_output
            ).float()

            original_image_features = l2_normalize(original_image_features)
            edited_image_features = l2_normalize(edited_image_features)
            edit_text_features = l2_normalize(edit_text_features)

            X_batch = torch.cat([original_image_features, edit_text_features], dim=-1).cpu()
            Y_batch = edited_image_features.cpu()

            X_chunks.append(X_batch)
            Y_chunks.append(Y_batch)

            sample_ids.extend([item["sample_id"] for item in batch])
            original_paths.extend([item["original_path"] for item in batch])
            edited_paths.extend([item["edited_path"] for item in batch])
            instructions.extend(edit_texts)

    X = torch.cat(X_chunks, dim=0).contiguous()
    Y = torch.cat(Y_chunks, dim=0).contiguous()

    payload = {
        "X": X,
        "Y": Y,
        "sample_ids": sample_ids,
        "original_paths": original_paths,
        "edited_paths": edited_paths,
        "instructions": instructions,
        "clip_model_id": CLIP_MODEL_ID,
        "num_samples": int(X.shape[0]),
        "input_dim": int(X.shape[1]),
        "target_dim": int(Y.shape[1]),
    }

    torch.save(payload, FEATURE_CACHE_PATH)
    torch.save(payload, FEATURE_CACHE_PATH_ALT1)
    torch.save(payload, FEATURE_CACHE_PATH_ALT2)

    meta = {
        "feature_cache_path": str(FEATURE_CACHE_PATH),
        "feature_cache_path_alt1": str(FEATURE_CACHE_PATH_ALT1),
        "feature_cache_path_alt2": str(FEATURE_CACHE_PATH_ALT2),
        "clip_model_id": CLIP_MODEL_ID,
        "num_samples": int(X.shape[0]),
        "input_dim": int(X.shape[1]),
        "target_dim": int(Y.shape[1]),
        "dataset_root": str(DATASET_ROOT),
        "index_path": str(INDEX_PATH),
        "batch_size": BATCH_SIZE,
        "device": device,
    }

    with open(FEATURE_META_PATH, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    # Make common names available to Cell 22.
    BRIDGE_FEATURE_CACHE_PATH = FEATURE_CACHE_PATH
    SUPERVISED_FEATURE_CACHE_PATH = FEATURE_CACHE_PATH
    ADAPTER_FEATURE_CACHE_PATH = FEATURE_CACHE_PATH
    FEATURE_CACHE_PT = FEATURE_CACHE_PATH

    print(f"Saved features to: {FEATURE_CACHE_PATH}")
    print(f"X shape: {tuple(X.shape)}")
    print(f"Y shape: {tuple(Y.shape)}")

Cell 21 device: cuda
Cell 21 samples: 1015


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting bridge features:   0%|          | 0/64 [00:00<?, ?it/s]

Saved features to: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/supervised_clip_features.pt
X shape: (1015, 1536)
Y shape: (1015, 768)


Cell 22

In [31]:
# Cell 22 — train and verify a residual feature bridge
# Predict edited_feature ≈ normalize(original_feature + delta(original_feature, edit_text_feature))

from pathlib import Path
import json
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 22: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Cell 22 device:", device)

    adapter_out_dir = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    adapter_out_dir.mkdir(parents=True, exist_ok=True)

    canonical_cache_dir = adapter_out_dir / "feature_cache"
    canonical_cache_dir.mkdir(parents=True, exist_ok=True)
    canonical_cache_path = canonical_cache_dir / "clip_feature_cache.pt"

    candidate_paths = []

    for key in [
        "BRIDGE_FEATURE_CACHE_PATH",
        "SUPERVISED_FEATURE_CACHE_PATH",
        "ADAPTER_FEATURE_CACHE_PATH",
        "FEATURE_CACHE_PT",
    ]:
        p = globals().get(key, None)
        if p is not None:
            candidate_paths.append(Path(p))

    candidate_paths.extend([
        canonical_cache_path,
        adapter_out_dir / "supervised_clip_features.pt",
        adapter_out_dir / "bridge_features.pt",
        adapter_out_dir / "adapter_features.pt",
    ])

    feature_cache_path = None
    seen = set()
    for p in candidate_paths:
        p = Path(p)
        if str(p) in seen:
            continue
        seen.add(str(p))
        if p.exists():
            feature_cache_path = p
            break

    if feature_cache_path is None:
        raise FileNotFoundError(
            "Missing feature cache from Cell 21. Checked:\n" +
            "\n".join(str(p) for p in candidate_paths)
        )

    print("Using feature cache:", feature_cache_path)

    payload = torch.load(feature_cache_path, map_location="cpu")

    if feature_cache_path != canonical_cache_path:
        torch.save(payload, canonical_cache_path)
        print("Canonicalized feature cache to:", canonical_cache_path)

    X = payload["X"].float().contiguous()
    Y = payload["Y"].float().contiguous()

    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError(f"Expected 2D X/Y tensors, got X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"Mismatched sample count: X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[1] != 2 * Y.shape[1]:
        raise ValueError(
            f"Residual bridge expects X dim == 2 * Y dim. "
            f"Got X dim {X.shape[1]}, Y dim {Y.shape[1]}"
        )

    num_samples = X.shape[0]
    input_dim = X.shape[1]
    target_dim = Y.shape[1]
    original_dim = target_dim
    text_dim = input_dim - original_dim

    print(f"Cell 22 X shape: {tuple(X.shape)}")
    print(f"Cell 22 Y shape: {tuple(Y.shape)}")

    SEED = int(globals().get("ADAPTER_SEED", 42))
    VAL_FRACTION = float(globals().get("ADAPTER_VAL_FRACTION", 0.2))
    BATCH_SIZE = int(globals().get("ADAPTER_BATCH_SIZE", 64))
    EPOCHS = int(globals().get("ADAPTER_EPOCHS", 40))
    LR = float(globals().get("ADAPTER_LR", 1e-3))
    WEIGHT_DECAY = float(globals().get("ADAPTER_WEIGHT_DECAY", 1e-4))
    DROPOUT = float(globals().get("ADAPTER_DROPOUT", 0.10))
    HIDDEN_DIM = int(globals().get("ADAPTER_HIDDEN_DIM", 2048))
    PATIENCE = int(globals().get("ADAPTER_PATIENCE", 8))

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    g = torch.Generator().manual_seed(SEED)
    perm = torch.randperm(num_samples, generator=g)

    val_size = max(1, int(round(num_samples * VAL_FRACTION)))
    val_size = min(val_size, num_samples - 1)
    train_size = num_samples - val_size

    train_idx = perm[:train_size]
    val_idx = perm[train_size:]

    X_train = X[train_idx]
    Y_train = Y[train_idx]
    X_val = X[val_idx]
    Y_val = Y[val_idx]

    print(f"Train samples: {len(train_idx)}")
    print(f"Val samples:   {len(val_idx)}")

    train_loader = DataLoader(
        TensorDataset(X_train, Y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
    )

    class ResidualFeatureBridge(nn.Module):
        def __init__(self, original_dim, text_dim, hidden_dim, dropout=0.1):
            super().__init__()
            in_dim = original_dim + text_dim

            self.backbone = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )

            self.delta_head = nn.Linear(hidden_dim, original_dim)
            self.gate_head = nn.Linear(hidden_dim, 1)

            # Start exactly at the baseline:
            # edited_pred = normalize(original + 0 * delta)
            nn.init.zeros_(self.delta_head.weight)
            nn.init.zeros_(self.delta_head.bias)
            nn.init.zeros_(self.gate_head.weight)
            nn.init.zeros_(self.gate_head.bias)

        def forward(self, x):
            original = x[:, :original_dim]
            h = self.backbone(x)
            delta = self.delta_head(h)
            gate = torch.sigmoid(self.gate_head(h))   # [B, 1]
            pred = original + gate * delta
            pred = F.normalize(pred, dim=-1)
            return pred, gate, delta

    model = ResidualFeatureBridge(
        original_dim=original_dim,
        text_dim=text_dim,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    def cosine_loss(pred, target):
        pred = F.normalize(pred, dim=-1)
        target = F.normalize(target, dim=-1)
        return 1.0 - (pred * target).sum(dim=-1).mean()

    @torch.inference_mode()
    def evaluate(model, X_eval, Y_eval):
        model.eval()

        X_eval = X_eval.to(device, non_blocking=True)
        Y_eval = F.normalize(Y_eval.to(device, non_blocking=True), dim=-1)

        pred, gate, delta = model(X_eval)
        pred = F.normalize(pred, dim=-1)

        loss = cosine_loss(pred, Y_eval).item()
        mean_cos = (pred * Y_eval).sum(dim=-1).mean().item()

        sim = pred @ Y_eval.T
        top1 = (sim.argmax(dim=1) == torch.arange(sim.shape[0], device=sim.device)).float().mean().item()

        baseline_pred = F.normalize(X_eval[:, :target_dim], dim=-1)
        baseline_mean_cos = (baseline_pred * Y_eval).sum(dim=-1).mean().item()
        baseline_sim = baseline_pred @ Y_eval.T
        baseline_top1 = (
            (baseline_sim.argmax(dim=1) == torch.arange(baseline_sim.shape[0], device=baseline_sim.device))
            .float()
            .mean()
            .item()
        )
        baseline_loss = cosine_loss(baseline_pred, Y_eval).item()

        delta_norm = delta.norm(dim=-1).mean().item()
        gate_mean = gate.mean().item()

        return {
            "loss": float(loss),
            "mean_cos": float(mean_cos),
            "top1_retrieval": float(top1),
            "baseline_loss": float(baseline_loss),
            "baseline_mean_cos": float(baseline_mean_cos),
            "baseline_top1_retrieval": float(baseline_top1),
            "delta_norm_mean": float(delta_norm),
            "gate_mean": float(gate_mean),
        }

    ckpt_dir = adapter_out_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = ckpt_dir / "feature_bridge_residual_best.pt"
    last_ckpt_path = ckpt_dir / "feature_bridge_residual_last.pt"
    metrics_path = adapter_out_dir / "feature_bridge_residual_metrics.json"

    history = []
    best_state = None
    best_val_loss = math.inf
    best_epoch = -1
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        count = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
        for xb, yb in pbar:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred, gate, delta = model(xb)
            main_loss = cosine_loss(pred, yb)

            # Light regularization to avoid giant arbitrary deltas.
            delta_penalty = 1e-4 * (delta.pow(2).mean())
            loss = main_loss + delta_penalty

            loss.backward()
            optimizer.step()

            bs = xb.shape[0]
            running_loss += main_loss.item() * bs
            count += bs
            pbar.set_postfix(train_loss=f"{running_loss / max(count, 1):.4f}")

        train_loss = running_loss / max(count, 1)
        val_metrics = evaluate(model, X_val, Y_val)

        row = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            **val_metrics,
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_cos={val_metrics['mean_cos']:.4f} | "
            f"val_top1={val_metrics['top1_retrieval']:.4f} | "
            f"baseline_cos={val_metrics['baseline_mean_cos']:.4f} | "
            f"baseline_top1={val_metrics['baseline_top1_retrieval']:.4f} | "
            f"gate_mean={val_metrics['gate_mean']:.4f} | "
            f"delta_norm={val_metrics['delta_norm_mean']:.4f}"
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "history": history,
                "input_dim": input_dim,
                "target_dim": target_dim,
                "hidden_dim": HIDDEN_DIM,
                "dropout": DROPOUT,
                "feature_cache_path": str(feature_cache_path),
                "model_type": "residual_feature_bridge",
            },
            last_ckpt_path,
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": best_state,
                    "history": history,
                    "input_dim": input_dim,
                    "target_dim": target_dim,
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                    "feature_cache_path": str(feature_cache_path),
                    "model_type": "residual_feature_bridge",
                },
                best_ckpt_path,
            )
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (patience={PATIENCE})")
                break

    if best_state is None:
        raise RuntimeError("Training ended without producing a best checkpoint.")

    model.load_state_dict(best_state)
    final_metrics = evaluate(model, X_val, Y_val)

    summary = {
        "best_epoch": int(best_epoch),
        "num_samples": int(num_samples),
        "train_samples": int(train_size),
        "val_samples": int(val_size),
        "input_dim": int(input_dim),
        "target_dim": int(target_dim),
        "original_dim": int(original_dim),
        "text_dim": int(text_dim),
        "hidden_dim": int(HIDDEN_DIM),
        "dropout": float(DROPOUT),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "batch_size": int(BATCH_SIZE),
        "epochs_requested": int(EPOCHS),
        "patience": int(PATIENCE),
        "feature_cache_path": str(feature_cache_path),
        "best_checkpoint_path": str(best_ckpt_path),
        "last_checkpoint_path": str(last_ckpt_path),
        "final_val_loss": float(final_metrics["loss"]),
        "final_val_mean_cosine": float(final_metrics["mean_cos"]),
        "final_val_top1_retrieval": float(final_metrics["top1_retrieval"]),
        "baseline_val_loss": float(final_metrics["baseline_loss"]),
        "baseline_val_mean_cosine": float(final_metrics["baseline_mean_cos"]),
        "baseline_val_top1_retrieval": float(final_metrics["baseline_top1_retrieval"]),
        "final_gate_mean": float(final_metrics["gate_mean"]),
        "final_delta_norm_mean": float(final_metrics["delta_norm_mean"]),
        "history": history,
        "model_type": "residual_feature_bridge",
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nCell 22 final metrics")
    print(f"best_epoch: {summary['best_epoch']}")
    print(f"val_loss: {summary['final_val_loss']:.6f}")
    print(f"val_mean_cosine: {summary['final_val_mean_cosine']:.6f}")
    print(f"val_top1_retrieval: {summary['final_val_top1_retrieval']:.6f}")
    print(f"baseline_val_loss: {summary['baseline_val_loss']:.6f}")
    print(f"baseline_val_mean_cosine: {summary['baseline_val_mean_cosine']:.6f}")
    print(f"baseline_val_top1_retrieval: {summary['baseline_val_top1_retrieval']:.6f}")
    print(f"final_gate_mean: {summary['final_gate_mean']:.6f}")
    print(f"final_delta_norm_mean: {summary['final_delta_norm_mean']:.6f}")
    print(f"metrics_json: {metrics_path}")
    print(f"best_checkpoint: {best_ckpt_path}")

Cell 22 device: cuda
Using feature cache: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/supervised_clip_features.pt
Canonicalized feature cache to: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/feature_cache/clip_feature_cache.pt
Cell 22 X shape: (1015, 1536)
Cell 22 Y shape: (1015, 768)
Train samples: 913
Val samples:   102


Epoch 1/12:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 01 | train_loss=0.1187 | val_loss=0.1066 | val_cos=0.8934 | val_top1=0.9118 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.4588 | delta_norm=0.6150


Epoch 2/12:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 02 | train_loss=0.1100 | val_loss=0.1072 | val_cos=0.8928 | val_top1=0.9020 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.4906 | delta_norm=0.6079


...

Epoch 07 | train_loss=0.0821 | val_loss=0.1156 | val_cos=0.8844 | val_top1=0.8824 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.6135 | delta_norm=0.6506


Epoch 8/12:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 08 | train_loss=0.0771 | val_loss=0.1194 | val_cos=0.8806 | val_top1=0.8824 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.6395 | delta_norm=0.6586


Epoch 9/12:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 09 | train_loss=0.0733 | val_loss=0.1216 | val_cos=0.8784 | val_top1=0.8922 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.6359 | delta_norm=0.5970
Early stopping at epoch 9 (patience=8)

Cell 22 final metrics
best_epoch: 1
val_loss: 0.106574
val_mean_cosine: 0.893426
val_top1_retrieval: 0.911765
baseline_val_loss: 0.112389
baseline_val_mean_cosine: 0.887611
baseline_val_top1_retrieval: 0.941177
final_gate_mean: 0.458775
final_delta_norm_mean: 0.615038
metrics_json: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/feature_bridge_residual_metrics.json
best_checkpoint: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/checkpoints/feature_bridge_residual_best.pt


Cell 22.1

In [32]:
# Cell 22.1 — multi-seed residual bridge evaluation

from pathlib import Path
import json
import math
import copy
import statistics
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 22.1: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Cell 22.1 device:", device)

    adapter_out_dir = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    adapter_out_dir.mkdir(parents=True, exist_ok=True)

    canonical_cache_path = adapter_out_dir / "feature_cache" / "clip_feature_cache.pt"
    candidate_paths = [
        globals().get("BRIDGE_FEATURE_CACHE_PATH", None),
        globals().get("SUPERVISED_FEATURE_CACHE_PATH", None),
        globals().get("ADAPTER_FEATURE_CACHE_PATH", None),
        globals().get("FEATURE_CACHE_PT", None),
        canonical_cache_path,
        adapter_out_dir / "supervised_clip_features.pt",
        adapter_out_dir / "bridge_features.pt",
        adapter_out_dir / "adapter_features.pt",
    ]

    feature_cache_path = None
    seen = set()
    for p in candidate_paths:
        if p is None:
            continue
        p = Path(p)
        if str(p) in seen:
            continue
        seen.add(str(p))
        if p.exists():
            feature_cache_path = p
            break

    if feature_cache_path is None:
        raise FileNotFoundError("Cell 22.1 could not find the feature cache from Cell 21.")

    print("Using feature cache:", feature_cache_path)
    payload = torch.load(feature_cache_path, map_location="cpu")

    X = payload["X"].float().contiguous()
    Y = payload["Y"].float().contiguous()

    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError(f"Expected 2D X/Y tensors, got X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"Mismatched sample count: X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[1] != 2 * Y.shape[1]:
        raise ValueError(
            f"Residual bridge expects X dim == 2 * Y dim. Got X dim {X.shape[1]}, Y dim {Y.shape[1]}"
        )

    num_samples = X.shape[0]
    input_dim = X.shape[1]
    target_dim = Y.shape[1]
    original_dim = target_dim
    text_dim = input_dim - original_dim

    print(f"Cell 22.1 X shape: {tuple(X.shape)}")
    print(f"Cell 22.1 Y shape: {tuple(Y.shape)}")

    # Multi-seed settings
    MULTISEED_SEEDS = list(globals().get("MULTISEED_SEEDS", [7, 17, 42, 99, 123]))
    VAL_FRACTION = float(globals().get("ADAPTER_VAL_FRACTION", 0.2))
    BATCH_SIZE = int(globals().get("ADAPTER_BATCH_SIZE", 64))
    HIDDEN_DIM = int(globals().get("ADAPTER_HIDDEN_DIM", 2048))
    DROPOUT = float(globals().get("ADAPTER_DROPOUT", 0.10))
    LR = float(globals().get("ADAPTER_LR", 1e-3))
    WEIGHT_DECAY = float(globals().get("ADAPTER_WEIGHT_DECAY", 1e-4))

    # Keep this somewhat lighter than Cell 22 for faster repeated evaluation.
    MULTISEED_EPOCHS = int(globals().get("MULTISEED_EPOCHS", min(int(globals().get("ADAPTER_EPOCHS", 40)), 15)))
    MULTISEED_PATIENCE = int(globals().get("MULTISEED_PATIENCE", 5))
    DELTA_PENALTY_WEIGHT = float(globals().get("RESIDUAL_DELTA_PENALTY_WEIGHT", 1e-4))

    class ResidualFeatureBridge(nn.Module):
        def __init__(self, original_dim, text_dim, hidden_dim, dropout=0.1):
            super().__init__()
            in_dim = original_dim + text_dim

            self.backbone = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.delta_head = nn.Linear(hidden_dim, original_dim)
            self.gate_head = nn.Linear(hidden_dim, 1)

            nn.init.zeros_(self.delta_head.weight)
            nn.init.zeros_(self.delta_head.bias)
            nn.init.zeros_(self.gate_head.weight)
            nn.init.zeros_(self.gate_head.bias)

        def forward(self, x):
            original = x[:, :original_dim]
            h = self.backbone(x)
            delta = self.delta_head(h)
            gate = torch.sigmoid(self.gate_head(h))
            pred = original + gate * delta
            pred = F.normalize(pred, dim=-1)
            return pred, gate, delta

    def cosine_loss(pred, target):
        pred = F.normalize(pred, dim=-1)
        target = F.normalize(target, dim=-1)
        return 1.0 - (pred * target).sum(dim=-1).mean()

    @torch.inference_mode()
    def evaluate(model, X_eval, Y_eval):
        model.eval()

        X_eval = X_eval.to(device, non_blocking=True)
        Y_eval = F.normalize(Y_eval.to(device, non_blocking=True), dim=-1)

        pred, gate, delta = model(X_eval)
        pred = F.normalize(pred, dim=-1)

        loss = cosine_loss(pred, Y_eval).item()
        mean_cos = (pred * Y_eval).sum(dim=-1).mean().item()

        sim = pred @ Y_eval.T
        top1 = (sim.argmax(dim=1) == torch.arange(sim.shape[0], device=sim.device)).float().mean().item()

        baseline_pred = F.normalize(X_eval[:, :target_dim], dim=-1)
        baseline_loss = cosine_loss(baseline_pred, Y_eval).item()
        baseline_mean_cos = (baseline_pred * Y_eval).sum(dim=-1).mean().item()
        baseline_sim = baseline_pred @ Y_eval.T
        baseline_top1 = (
            (baseline_sim.argmax(dim=1) == torch.arange(baseline_sim.shape[0], device=baseline_sim.device))
            .float()
            .mean()
            .item()
        )

        return {
            "loss": float(loss),
            "mean_cos": float(mean_cos),
            "top1_retrieval": float(top1),
            "baseline_loss": float(baseline_loss),
            "baseline_mean_cos": float(baseline_mean_cos),
            "baseline_top1_retrieval": float(baseline_top1),
            "delta_norm_mean": float(delta.norm(dim=-1).mean().item()),
            "gate_mean": float(gate.mean().item()),
        }

    def train_one_seed(seed: int):
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        g = torch.Generator().manual_seed(seed)
        perm = torch.randperm(num_samples, generator=g)

        val_size = max(1, int(round(num_samples * VAL_FRACTION)))
        val_size = min(val_size, num_samples - 1)
        train_size = num_samples - val_size

        train_idx = perm[:train_size]
        val_idx = perm[train_size:]

        X_train = X[train_idx]
        Y_train = Y[train_idx]
        X_val = X[val_idx]
        Y_val = Y[val_idx]

        train_loader = DataLoader(
            TensorDataset(X_train, Y_train),
            batch_size=BATCH_SIZE,
            shuffle=True,
            drop_last=False,
            pin_memory=torch.cuda.is_available(),
        )

        model = ResidualFeatureBridge(
            original_dim=original_dim,
            text_dim=text_dim,
            hidden_dim=HIDDEN_DIM,
            dropout=DROPOUT,
        ).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )

        best_state = None
        best_val_loss = math.inf
        best_epoch = -1
        no_improve = 0

        for epoch in range(1, MULTISEED_EPOCHS + 1):
            model.train()
            for xb, yb in train_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)
                pred, gate, delta = model(xb)
                main_loss = cosine_loss(pred, yb)
                delta_penalty = DELTA_PENALTY_WEIGHT * delta.pow(2).mean()
                loss = main_loss + delta_penalty
                loss.backward()
                optimizer.step()

            val_metrics = evaluate(model, X_val, Y_val)

            if val_metrics["loss"] < best_val_loss:
                best_val_loss = val_metrics["loss"]
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= MULTISEED_PATIENCE:
                    break

        if best_state is None:
            raise RuntimeError(f"Seed {seed}: no best state captured.")

        model.load_state_dict(best_state)
        final_metrics = evaluate(model, X_val, Y_val)

        return {
            "seed": int(seed),
            "best_epoch": int(best_epoch),
            "train_samples": int(train_size),
            "val_samples": int(val_size),
            "val_loss": float(final_metrics["loss"]),
            "val_mean_cosine": float(final_metrics["mean_cos"]),
            "val_top1_retrieval": float(final_metrics["top1_retrieval"]),
            "baseline_val_loss": float(final_metrics["baseline_loss"]),
            "baseline_val_mean_cosine": float(final_metrics["baseline_mean_cos"]),
            "baseline_val_top1_retrieval": float(final_metrics["baseline_top1_retrieval"]),
            "improvement_loss": float(final_metrics["baseline_loss"] - final_metrics["loss"]),
            "improvement_mean_cosine": float(final_metrics["mean_cos"] - final_metrics["baseline_mean_cos"]),
            "improvement_top1_retrieval": float(final_metrics["top1_retrieval"] - final_metrics["baseline_top1_retrieval"]),
            "gate_mean": float(final_metrics["gate_mean"]),
            "delta_norm_mean": float(final_metrics["delta_norm_mean"]),
        }

    results = []
    for seed in MULTISEED_SEEDS:
        print(f"\nRunning seed {seed} ...")
        result = train_one_seed(int(seed))
        results.append(result)
        print(
            f"seed={result['seed']} | "
            f"best_epoch={result['best_epoch']} | "
            f"val_cos={result['val_mean_cosine']:.6f} vs baseline={result['baseline_val_mean_cosine']:.6f} | "
            f"val_top1={result['val_top1_retrieval']:.6f} vs baseline={result['baseline_val_top1_retrieval']:.6f} | "
            f"delta_cos={result['improvement_mean_cosine']:.6f} | "
            f"delta_top1={result['improvement_top1_retrieval']:.6f}"
        )

    def mean_std(key):
        vals = [r[key] for r in results]
        mean_val = statistics.fmean(vals)
        std_val = statistics.stdev(vals) if len(vals) > 1 else 0.0
        return float(mean_val), float(std_val)

    summary = {
        "model_type": "residual_feature_bridge_multiseed",
        "feature_cache_path": str(feature_cache_path),
        "num_samples": int(num_samples),
        "input_dim": int(input_dim),
        "target_dim": int(target_dim),
        "seeds": [int(s) for s in MULTISEED_SEEDS],
        "epochs_per_seed": int(MULTISEED_EPOCHS),
        "patience_per_seed": int(MULTISEED_PATIENCE),
        "batch_size": int(BATCH_SIZE),
        "hidden_dim": int(HIDDEN_DIM),
        "dropout": float(DROPOUT),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "val_fraction": float(VAL_FRACTION),
        "results": results,
        "aggregate": {
            "val_loss_mean": mean_std("val_loss")[0],
            "val_loss_std": mean_std("val_loss")[1],
            "val_mean_cosine_mean": mean_std("val_mean_cosine")[0],
            "val_mean_cosine_std": mean_std("val_mean_cosine")[1],
            "val_top1_retrieval_mean": mean_std("val_top1_retrieval")[0],
            "val_top1_retrieval_std": mean_std("val_top1_retrieval")[1],
            "baseline_val_loss_mean": mean_std("baseline_val_loss")[0],
            "baseline_val_loss_std": mean_std("baseline_val_loss")[1],
            "baseline_val_mean_cosine_mean": mean_std("baseline_val_mean_cosine")[0],
            "baseline_val_mean_cosine_std": mean_std("baseline_val_mean_cosine")[1],
            "baseline_val_top1_retrieval_mean": mean_std("baseline_val_top1_retrieval")[0],
            "baseline_val_top1_retrieval_std": mean_std("baseline_val_top1_retrieval")[1],
            "improvement_loss_mean": mean_std("improvement_loss")[0],
            "improvement_loss_std": mean_std("improvement_loss")[1],
            "improvement_mean_cosine_mean": mean_std("improvement_mean_cosine")[0],
            "improvement_mean_cosine_std": mean_std("improvement_mean_cosine")[1],
            "improvement_top1_retrieval_mean": mean_std("improvement_top1_retrieval")[0],
            "improvement_top1_retrieval_std": mean_std("improvement_top1_retrieval")[1],
            "gate_mean_mean": mean_std("gate_mean")[0],
            "gate_mean_std": mean_std("gate_mean")[1],
            "delta_norm_mean_mean": mean_std("delta_norm_mean")[0],
            "delta_norm_mean_std": mean_std("delta_norm_mean")[1],
            "wins_on_cosine": int(sum(r["improvement_mean_cosine"] > 0 for r in results)),
            "wins_on_top1": int(sum(r["improvement_top1_retrieval"] > 0 for r in results)),
            "wins_on_both": int(sum((r["improvement_mean_cosine"] > 0) and (r["improvement_top1_retrieval"] > 0) for r in results)),
        },
    }

    multiseed_metrics_path = adapter_out_dir / "feature_bridge_residual_multiseed_metrics.json"
    with open(multiseed_metrics_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    agg = summary["aggregate"]

    print("\nCell 22.1 aggregate metrics")
    print(f"seeds: {summary['seeds']}")
    print(f"val_mean_cosine mean±std: {agg['val_mean_cosine_mean']:.6f} ± {agg['val_mean_cosine_std']:.6f}")
    print(f"baseline_mean_cosine mean±std: {agg['baseline_val_mean_cosine_mean']:.6f} ± {agg['baseline_val_mean_cosine_std']:.6f}")
    print(f"improvement_mean_cosine mean±std: {agg['improvement_mean_cosine_mean']:.6f} ± {agg['improvement_mean_cosine_std']:.6f}")
    print(f"val_top1 mean±std: {agg['val_top1_retrieval_mean']:.6f} ± {agg['val_top1_retrieval_std']:.6f}")
    print(f"baseline_top1 mean±std: {agg['baseline_val_top1_retrieval_mean']:.6f} ± {agg['baseline_val_top1_retrieval_std']:.6f}")
    print(f"improvement_top1 mean±std: {agg['improvement_top1_retrieval_mean']:.6f} ± {agg['improvement_top1_retrieval_std']:.6f}")
    print(f"wins_on_cosine: {agg['wins_on_cosine']}/{len(results)}")
    print(f"wins_on_top1: {agg['wins_on_top1']}/{len(results)}")
    print(f"wins_on_both: {agg['wins_on_both']}/{len(results)}")
    print(f"metrics_json: {multiseed_metrics_path}")

Cell 22.1 device: cuda
Using feature cache: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/supervised_clip_features.pt
Cell 22.1 X shape: (1015, 1536)
Cell 22.1 Y shape: (1015, 768)

Running seed 7 ...
seed=7 | best_epoch=1 | val_cos=0.887570 vs baseline=0.879261 | val_top1=0.911765 vs baseline=0.921569 | delta_cos=0.008309 | delta_top1=-0.009804

Running seed 17 ...
seed=17 | best_epoch=1 | val_cos=0.889231 vs baseline=0.880258 | val_top1=0.872549 vs baseline=0.872549 | delta_cos=0.008973 | delta_top1=0.000000

Running seed 42 ...
seed=42 | best_epoch=1 | val_cos=0.893894 vs baseline=0.887611 | val_top1=0.911765 vs baseline=0.941177 | delta_cos=0.006283 | delta_top1=-0.029412

Running seed 99 ...
seed=99 | best_epoch=1 | val_cos=0.884005 vs baseline=0.877090 | val_top1=0.872549 vs baseline=0.911765 | delta_cos=0.006915 | delta_top1=-0.039216

Running seed 123 ...
seed=123 | best_epoch=2 | val_cos=0.883670 vs baseline=0.872941 | val_top1=0.941177 vs baseline=0

Cell 22.2

In [33]:
# Cell 22.2 — train a retrieval-aware residual bridge
# Goal: keep the residual setup, but optimize ranking/retrieval as well as cosine alignment.

from pathlib import Path
import json
import math
import copy
import statistics
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 22.2: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Cell 22.2 device:", device)

    adapter_out_dir = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    adapter_out_dir.mkdir(parents=True, exist_ok=True)

    feature_cache_candidates = [
        globals().get("BRIDGE_FEATURE_CACHE_PATH", None),
        globals().get("SUPERVISED_FEATURE_CACHE_PATH", None),
        globals().get("ADAPTER_FEATURE_CACHE_PATH", None),
        globals().get("FEATURE_CACHE_PT", None),
        adapter_out_dir / "feature_cache" / "clip_feature_cache.pt",
        adapter_out_dir / "supervised_clip_features.pt",
        adapter_out_dir / "bridge_features.pt",
        adapter_out_dir / "adapter_features.pt",
    ]

    feature_cache_path = None
    seen = set()
    for p in feature_cache_candidates:
        if p is None:
            continue
        p = Path(p)
        if str(p) in seen:
            continue
        seen.add(str(p))
        if p.exists():
            feature_cache_path = p
            break

    if feature_cache_path is None:
        raise FileNotFoundError("Cell 22.2 could not find the feature cache from Cell 21.")

    print("Using feature cache:", feature_cache_path)
    payload = torch.load(feature_cache_path, map_location="cpu")

    X = payload["X"].float().contiguous()
    Y = payload["Y"].float().contiguous()

    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError(f"Expected 2D X/Y tensors, got X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"Mismatched sample count: X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[1] != 2 * Y.shape[1]:
        raise ValueError(
            f"Retrieval-aware residual bridge expects X dim == 2 * Y dim. "
            f"Got X dim {X.shape[1]}, Y dim {Y.shape[1]}"
        )

    num_samples = X.shape[0]
    input_dim = X.shape[1]
    target_dim = Y.shape[1]
    original_dim = target_dim
    text_dim = input_dim - original_dim

    print(f"Cell 22.2 X shape: {tuple(X.shape)}")
    print(f"Cell 22.2 Y shape: {tuple(Y.shape)}")

    # Settings
    SEED = int(globals().get("ADAPTER_SEED", 42))
    VAL_FRACTION = float(globals().get("ADAPTER_VAL_FRACTION", 0.10))
    BATCH_SIZE = int(globals().get("ADAPTER_BATCH_SIZE", 64))
    EPOCHS = int(globals().get("RETRIEVAL_ADAPTER_EPOCHS", 20))
    LR = float(globals().get("RETRIEVAL_ADAPTER_LR", 5e-5))
    WEIGHT_DECAY = float(globals().get("ADAPTER_WEIGHT_DECAY", 1e-4))
    DROPOUT = float(globals().get("ADAPTER_DROPOUT", 0.10))
    HIDDEN_DIM = int(globals().get("ADAPTER_HIDDEN_DIM", 2048))
    PATIENCE = int(globals().get("RETRIEVAL_ADAPTER_PATIENCE", 6))

    # Loss weights
    COSINE_LOSS_WEIGHT = float(globals().get("RETRIEVAL_COSINE_LOSS_WEIGHT", 0.35))
    INFO_NCE_LOSS_WEIGHT = float(globals().get("RETRIEVAL_INFO_NCE_LOSS_WEIGHT", 1.00))
    DELTA_PENALTY_WEIGHT = float(globals().get("RETRIEVAL_DELTA_PENALTY_WEIGHT", 5e-5))
    TEMPERATURE = float(globals().get("RETRIEVAL_TEMPERATURE", 0.07))

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    g = torch.Generator().manual_seed(SEED)
    perm = torch.randperm(num_samples, generator=g)

    val_size = max(1, int(round(num_samples * VAL_FRACTION)))
    val_size = min(val_size, num_samples - 1)
    train_size = num_samples - val_size

    train_idx = perm[:train_size]
    val_idx = perm[train_size:]

    X_train = X[train_idx]
    Y_train = Y[train_idx]
    X_val = X[val_idx]
    Y_val = Y[val_idx]

    print(f"Train samples: {len(train_idx)}")
    print(f"Val samples:   {len(val_idx)}")

    train_loader = DataLoader(
        TensorDataset(X_train, Y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
    )

    class RetrievalAwareResidualBridge(nn.Module):
        def __init__(self, original_dim, text_dim, hidden_dim, dropout=0.1):
            super().__init__()
            in_dim = original_dim + text_dim

            self.backbone = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.delta_head = nn.Linear(hidden_dim, original_dim)
            self.gate_head = nn.Linear(hidden_dim, 1)

            nn.init.zeros_(self.delta_head.weight)
            nn.init.zeros_(self.delta_head.bias)
            nn.init.zeros_(self.gate_head.weight)
            nn.init.zeros_(self.gate_head.bias)

        def forward(self, x):
            original = x[:, :original_dim]
            h = self.backbone(x)
            delta = self.delta_head(h)
            gate = torch.sigmoid(self.gate_head(h))
            pred = original + gate * delta
            pred = F.normalize(pred, dim=-1)
            return pred, gate, delta

    model = RetrievalAwareResidualBridge(
        original_dim=original_dim,
        text_dim=text_dim,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    def cosine_loss(pred, target):
        pred = F.normalize(pred, dim=-1)
        target = F.normalize(target, dim=-1)
        return 1.0 - (pred * target).sum(dim=-1).mean()

    def info_nce_loss(pred, target, temperature=0.07):
        pred = F.normalize(pred, dim=-1)
        target = F.normalize(target, dim=-1)
        logits = (pred @ target.T) / temperature
        labels = torch.arange(logits.shape[0], device=logits.device)
        return F.cross_entropy(logits, labels)

    @torch.inference_mode()
    def evaluate(model, X_eval, Y_eval):
        model.eval()

        X_eval = X_eval.to(device, non_blocking=True)
        Y_eval = F.normalize(Y_eval.to(device, non_blocking=True), dim=-1)

        pred, gate, delta = model(X_eval)
        pred = F.normalize(pred, dim=-1)

        loss_cos = cosine_loss(pred, Y_eval).item()
        mean_cos = (pred * Y_eval).sum(dim=-1).mean().item()

        sim = pred @ Y_eval.T
        top1 = (sim.argmax(dim=1) == torch.arange(sim.shape[0], device=sim.device)).float().mean().item()

        baseline_pred = F.normalize(X_eval[:, :target_dim], dim=-1)
        baseline_loss = cosine_loss(baseline_pred, Y_eval).item()
        baseline_mean_cos = (baseline_pred * Y_eval).sum(dim=-1).mean().item()
        baseline_sim = baseline_pred @ Y_eval.T
        baseline_top1 = (
            (baseline_sim.argmax(dim=1) == torch.arange(baseline_sim.shape[0], device=sim.device))
            .float()
            .mean()
            .item()
        )

        return {
            "loss": float(loss_cos),
            "mean_cos": float(mean_cos),
            "top1_retrieval": float(top1),
            "baseline_loss": float(baseline_loss),
            "baseline_mean_cos": float(baseline_mean_cos),
            "baseline_top1_retrieval": float(baseline_top1),
            "gate_mean": float(gate.mean().item()),
            "delta_norm_mean": float(delta.norm(dim=-1).mean().item()),
        }

    ckpt_dir = adapter_out_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = ckpt_dir / "feature_bridge_retrieval_best.pt"
    last_ckpt_path = ckpt_dir / "feature_bridge_retrieval_last.pt"
    metrics_path = adapter_out_dir / "feature_bridge_retrieval_metrics.json"

    history = []
    best_state = None
    best_epoch = -1
    best_score = -1.0
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_total = 0.0
        running_cos = 0.0
        running_nce = 0.0
        count = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
        for xb, yb in pbar:
            xb = xb.to(device, non_blocking=True)
            yb = F.normalize(yb.to(device, non_blocking=True), dim=-1)

            optimizer.zero_grad(set_to_none=True)

            pred, gate, delta = model(xb)

            loss_cos = cosine_loss(pred, yb)
            loss_nce = info_nce_loss(pred, yb, temperature=TEMPERATURE)
            delta_penalty = DELTA_PENALTY_WEIGHT * delta.pow(2).mean()

            loss = (
                COSINE_LOSS_WEIGHT * loss_cos
                + INFO_NCE_LOSS_WEIGHT * loss_nce
                + delta_penalty
            )

            loss.backward()
            optimizer.step()

            bs = xb.shape[0]
            running_total += loss.item() * bs
            running_cos += loss_cos.item() * bs
            running_nce += loss_nce.item() * bs
            count += bs

            pbar.set_postfix(
                total=f"{running_total / max(count, 1):.4f}",
                cos=f"{running_cos / max(count, 1):.4f}",
                nce=f"{running_nce / max(count, 1):.4f}",
            )

        val_metrics = evaluate(model, X_val, Y_val)

        row = {
            "epoch": epoch,
            "train_total_loss": float(running_total / max(count, 1)),
            "train_cosine_loss": float(running_cos / max(count, 1)),
            "train_info_nce_loss": float(running_nce / max(count, 1)),
            **val_metrics,
        }
        history.append(row)

        # Selection score: prioritize top1 first, cosine second.
        selection_score = val_metrics["top1_retrieval"] + 0.10 * val_metrics["mean_cos"]

        print(
            f"Epoch {epoch:02d} | "
            f"val_cos={val_metrics['mean_cos']:.4f} | "
            f"val_top1={val_metrics['top1_retrieval']:.4f} | "
            f"baseline_cos={val_metrics['baseline_mean_cos']:.4f} | "
            f"baseline_top1={val_metrics['baseline_top1_retrieval']:.4f} | "
            f"gate_mean={val_metrics['gate_mean']:.4f} | "
            f"delta_norm={val_metrics['delta_norm_mean']:.4f}"
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "history": history,
                "input_dim": input_dim,
                "target_dim": target_dim,
                "hidden_dim": HIDDEN_DIM,
                "dropout": DROPOUT,
                "feature_cache_path": str(feature_cache_path),
                "model_type": "retrieval_aware_residual_feature_bridge",
            },
            last_ckpt_path,
        )

        if selection_score > best_score:
            best_score = selection_score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": best_state,
                    "history": history,
                    "input_dim": input_dim,
                    "target_dim": target_dim,
                    "hidden_dim": HIDDEN_DIM,
                    "dropout": DROPOUT,
                    "feature_cache_path": str(feature_cache_path),
                    "model_type": "retrieval_aware_residual_feature_bridge",
                },
                best_ckpt_path,
            )
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"Early stopping at epoch {epoch} (patience={PATIENCE})")
                break

    if best_state is None:
        raise RuntimeError("Training ended without producing a best checkpoint.")

    model.load_state_dict(best_state)
    final_metrics = evaluate(model, X_val, Y_val)

    summary = {
        "best_epoch": int(best_epoch),
        "num_samples": int(num_samples),
        "train_samples": int(train_size),
        "val_samples": int(val_size),
        "input_dim": int(input_dim),
        "target_dim": int(target_dim),
        "original_dim": int(original_dim),
        "text_dim": int(text_dim),
        "hidden_dim": int(HIDDEN_DIM),
        "dropout": float(DROPOUT),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "batch_size": int(BATCH_SIZE),
        "epochs_requested": int(EPOCHS),
        "patience": int(PATIENCE),
        "temperature": float(TEMPERATURE),
        "cosine_loss_weight": float(COSINE_LOSS_WEIGHT),
        "info_nce_loss_weight": float(INFO_NCE_LOSS_WEIGHT),
        "delta_penalty_weight": float(DELTA_PENALTY_WEIGHT),
        "feature_cache_path": str(feature_cache_path),
        "best_checkpoint_path": str(best_ckpt_path),
        "last_checkpoint_path": str(last_ckpt_path),
        "final_val_loss": float(final_metrics["loss"]),
        "final_val_mean_cosine": float(final_metrics["mean_cos"]),
        "final_val_top1_retrieval": float(final_metrics["top1_retrieval"]),
        "baseline_val_loss": float(final_metrics["baseline_loss"]),
        "baseline_val_mean_cosine": float(final_metrics["baseline_mean_cos"]),
        "baseline_val_top1_retrieval": float(final_metrics["baseline_top1_retrieval"]),
        "final_gate_mean": float(final_metrics["gate_mean"]),
        "final_delta_norm_mean": float(final_metrics["delta_norm_mean"]),
        "history": history,
        "model_type": "retrieval_aware_residual_feature_bridge",
    }

    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\nCell 22.2 final metrics")
    print(f"best_epoch: {summary['best_epoch']}")
    print(f"val_loss: {summary['final_val_loss']:.6f}")
    print(f"val_mean_cosine: {summary['final_val_mean_cosine']:.6f}")
    print(f"val_top1_retrieval: {summary['final_val_top1_retrieval']:.6f}")
    print(f"baseline_val_loss: {summary['baseline_val_loss']:.6f}")
    print(f"baseline_val_mean_cosine: {summary['baseline_val_mean_cosine']:.6f}")
    print(f"baseline_val_top1_retrieval: {summary['baseline_val_top1_retrieval']:.6f}")
    print(f"final_gate_mean: {summary['final_gate_mean']:.6f}")
    print(f"final_delta_norm_mean: {summary['final_delta_norm_mean']:.6f}")
    print(f"metrics_json: {metrics_path}")
    print(f"best_checkpoint: {best_ckpt_path}")

Cell 22.2 device: cuda
Using feature cache: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/supervised_clip_features.pt
Cell 22.2 X shape: (1015, 1536)
Cell 22.2 Y shape: (1015, 768)
Train samples: 913
Val samples:   102


Epoch 1/20:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 01 | val_cos=0.7977 | val_top1=0.9804 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.5010 | delta_norm=0.7804


Epoch 2/20:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 02 | val_cos=0.8194 | val_top1=0.9902 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.5058 | delta_norm=0.6547


...

Epoch 06 | val_cos=0.8029 | val_top1=0.9902 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.5460 | delta_norm=0.6713


Epoch 7/20:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 07 | val_cos=0.7894 | val_top1=0.9804 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.5623 | delta_norm=0.7074


Epoch 8/20:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 08 | val_cos=0.8124 | val_top1=0.9804 | baseline_cos=0.8876 | baseline_top1=0.9412 | gate_mean=0.5626 | delta_norm=0.6227
Early stopping at epoch 8 (patience=6)

Cell 22.2 final metrics
best_epoch: 2
val_loss: 0.180612
val_mean_cosine: 0.819388
val_top1_retrieval: 0.990196
baseline_val_loss: 0.112389
baseline_val_mean_cosine: 0.887611
baseline_val_top1_retrieval: 0.941177
final_gate_mean: 0.505835
final_delta_norm_mean: 0.654669
metrics_json: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/feature_bridge_retrieval_metrics.json
best_checkpoint: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/checkpoints/feature_bridge_retrieval_best.pt


Cell 22.3

In [34]:
# Cell 22.3 — compare bridge variants and choose a deployment checkpoint

from pathlib import Path
import json

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 22.3: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    adapter_out_dir = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    adapter_out_dir.mkdir(parents=True, exist_ok=True)

    # Selection mode:
    #   "cosine"   -> prefer closest edited-feature regression
    #   "top1"     -> prefer retrieval/ranking performance
    #   "balanced" -> require positive top1 gain first, then prefer cosine among winners
    BRIDGE_SELECTION_MODE = globals().get("BRIDGE_SELECTION_MODE", "top1")

    residual_metrics_path = adapter_out_dir / "feature_bridge_residual_metrics.json"
    retrieval_metrics_path = adapter_out_dir / "feature_bridge_retrieval_metrics.json"
    selection_out_path = adapter_out_dir / "selected_bridge.json"

    if not residual_metrics_path.exists():
        raise FileNotFoundError(f"Missing residual metrics: {residual_metrics_path}")
    if not retrieval_metrics_path.exists():
        raise FileNotFoundError(f"Missing retrieval-aware metrics: {retrieval_metrics_path}")

    with open(residual_metrics_path, "r", encoding="utf-8") as f:
        residual = json.load(f)

    with open(retrieval_metrics_path, "r", encoding="utf-8") as f:
        retrieval = json.load(f)

    candidates = [
        {
            "name": "residual_feature_bridge",
            "metrics_path": str(residual_metrics_path),
            "checkpoint_path": residual["best_checkpoint_path"],
            "val_mean_cosine": residual["final_val_mean_cosine"],
            "val_top1_retrieval": residual["final_val_top1_retrieval"],
            "baseline_val_mean_cosine": residual["baseline_val_mean_cosine"],
            "baseline_val_top1_retrieval": residual["baseline_val_top1_retrieval"],
            "delta_mean_cosine": residual["final_val_mean_cosine"] - residual["baseline_val_mean_cosine"],
            "delta_top1_retrieval": residual["final_val_top1_retrieval"] - residual["baseline_val_top1_retrieval"],
            "model_type": residual.get("model_type", "residual_feature_bridge"),
        },
        {
            "name": "retrieval_aware_residual_feature_bridge",
            "metrics_path": str(retrieval_metrics_path),
            "checkpoint_path": retrieval["best_checkpoint_path"],
            "val_mean_cosine": retrieval["final_val_mean_cosine"],
            "val_top1_retrieval": retrieval["final_val_top1_retrieval"],
            "baseline_val_mean_cosine": retrieval["baseline_val_mean_cosine"],
            "baseline_val_top1_retrieval": retrieval["baseline_val_top1_retrieval"],
            "delta_mean_cosine": retrieval["final_val_mean_cosine"] - retrieval["baseline_val_mean_cosine"],
            "delta_top1_retrieval": retrieval["final_val_top1_retrieval"] - retrieval["baseline_val_top1_retrieval"],
            "model_type": retrieval.get("model_type", "retrieval_aware_residual_feature_bridge"),
        },
    ]

    if BRIDGE_SELECTION_MODE == "cosine":
        selected = max(
            candidates,
            key=lambda x: (x["delta_mean_cosine"], x["delta_top1_retrieval"])
        )

    elif BRIDGE_SELECTION_MODE == "top1":
        selected = max(
            candidates,
            key=lambda x: (x["delta_top1_retrieval"], x["delta_mean_cosine"])
        )

    elif BRIDGE_SELECTION_MODE == "balanced":
        positive_top1 = [c for c in candidates if c["delta_top1_retrieval"] > 0]
        if positive_top1:
            selected = max(
                positive_top1,
                key=lambda x: (x["delta_mean_cosine"], x["delta_top1_retrieval"])
            )
        else:
            selected = max(
                candidates,
                key=lambda x: (x["delta_top1_retrieval"], x["delta_mean_cosine"])
            )
    else:
        raise ValueError(f"Bad BRIDGE_SELECTION_MODE: {BRIDGE_SELECTION_MODE}")

    summary = {
        "bridge_selection_mode": BRIDGE_SELECTION_MODE,
        "selected_name": selected["name"],
        "selected_model_type": selected["model_type"],
        "selected_checkpoint_path": selected["checkpoint_path"],
        "selected_metrics_path": selected["metrics_path"],
        "selected_val_mean_cosine": selected["val_mean_cosine"],
        "selected_val_top1_retrieval": selected["val_top1_retrieval"],
        "selected_delta_mean_cosine": selected["delta_mean_cosine"],
        "selected_delta_top1_retrieval": selected["delta_top1_retrieval"],
        "candidates": candidates,
    }

    with open(selection_out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("Cell 22.3 bridge comparison")
    for c in candidates:
        print(
            f"{c['name']}: "
            f"delta_cos={c['delta_mean_cosine']:+.6f} | "
            f"delta_top1={c['delta_top1_retrieval']:+.6f} | "
            f"ckpt={c['checkpoint_path']}"
        )

    print("\nCell 22.3 selected bridge")
    print(f"BRIDGE_SELECTION_MODE: {summary['bridge_selection_mode']}")
    print(f"selected_name: {summary['selected_name']}")
    print(f"selected_checkpoint_path: {summary['selected_checkpoint_path']}")
    print(f"selected_delta_mean_cosine: {summary['selected_delta_mean_cosine']:+.6f}")
    print(f"selected_delta_top1_retrieval: {summary['selected_delta_top1_retrieval']:+.6f}")
    print(f"selection_json: {selection_out_path}")

Cell 22.3 bridge comparison
residual_feature_bridge: delta_cos=+0.005814 | delta_top1=-0.029412 | ckpt=/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/checkpoints/feature_bridge_residual_best.pt
retrieval_aware_residual_feature_bridge: delta_cos=-0.068223 | delta_top1=+0.049020 | ckpt=/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/checkpoints/feature_bridge_retrieval_best.pt

Cell 22.3 selected bridge
BRIDGE_SELECTION_MODE: top1
selected_name: retrieval_aware_residual_feature_bridge
selected_checkpoint_path: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/checkpoints/feature_bridge_retrieval_best.pt
selected_delta_mean_cosine: -0.068223
selected_delta_top1_retrieval: +0.049020
selection_json: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/selected_bridge.json


Cell 22.4

In [35]:
# Cell 22.4 — publish the selected bridge as the current deployment package

from pathlib import Path
import json
import shutil
import torch

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 22.4: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    adapter_out_dir = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    selection_json = adapter_out_dir / "selected_bridge.json"
    deploy_dir = adapter_out_dir / "deploy"
    deploy_dir.mkdir(parents=True, exist_ok=True)

    if not selection_json.exists():
        raise FileNotFoundError(f"Missing selected bridge file: {selection_json}")

    selection = json.loads(selection_json.read_text(encoding="utf-8"))
    selected_ckpt = Path(selection["selected_checkpoint_path"])
    selected_metrics_path = Path(selection["selected_metrics_path"])

    if not selected_ckpt.exists():
        raise FileNotFoundError(f"Selected checkpoint does not exist: {selected_ckpt}")
    if not selected_metrics_path.exists():
        raise FileNotFoundError(f"Selected metrics file does not exist: {selected_metrics_path}")

    metrics = json.loads(selected_metrics_path.read_text(encoding="utf-8"))
    ckpt = torch.load(selected_ckpt, map_location="cpu")

    deploy_ckpt_path = deploy_dir / "current_bridge.pt"
    deploy_metrics_path = deploy_dir / "current_bridge_metrics.json"
    deploy_manifest_path = deploy_dir / "current_bridge_manifest.json"

    shutil.copy2(selected_ckpt, deploy_ckpt_path)
    shutil.copy2(selected_metrics_path, deploy_metrics_path)

    manifest = {
        "bridge_selection_mode": selection["bridge_selection_mode"],
        "selected_name": selection["selected_name"],
        "selected_model_type": selection["selected_model_type"],
        "source_checkpoint_path": str(selected_ckpt),
        "source_metrics_path": str(selected_metrics_path),
        "deploy_checkpoint_path": str(deploy_ckpt_path),
        "deploy_metrics_path": str(deploy_metrics_path),
        "input_dim": ckpt.get("input_dim"),
        "target_dim": ckpt.get("target_dim"),
        "hidden_dim": ckpt.get("hidden_dim"),
        "dropout": ckpt.get("dropout"),
        "feature_cache_path": ckpt.get("feature_cache_path"),
        "selected_val_mean_cosine": selection["selected_val_mean_cosine"],
        "selected_val_top1_retrieval": selection["selected_val_top1_retrieval"],
        "selected_delta_mean_cosine": selection["selected_delta_mean_cosine"],
        "selected_delta_top1_retrieval": selection["selected_delta_top1_retrieval"],
    }

    deploy_manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    # Expose stable globals for later cells
    CURRENT_BRIDGE_CHECKPOINT = str(deploy_ckpt_path)
    CURRENT_BRIDGE_METRICS = str(deploy_metrics_path)
    CURRENT_BRIDGE_MANIFEST = str(deploy_manifest_path)
    CURRENT_BRIDGE_MODEL_TYPE = manifest["selected_model_type"]

    print("Cell 22.4 published bridge")
    print(f"selected_name: {manifest['selected_name']}")
    print(f"selected_model_type: {manifest['selected_model_type']}")
    print(f"deploy_checkpoint_path: {deploy_ckpt_path}")
    print(f"deploy_metrics_path: {deploy_metrics_path}")
    print(f"deploy_manifest_path: {deploy_manifest_path}")

Cell 22.4 published bridge
selected_name: retrieval_aware_residual_feature_bridge
selected_model_type: retrieval_aware_residual_feature_bridge
deploy_checkpoint_path: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/deploy/current_bridge.pt
deploy_metrics_path: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/deploy/current_bridge_metrics.json
deploy_manifest_path: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/deploy/current_bridge_manifest.json


Cell 22.5

In [36]:
# Cell 22.5 — smoke test the deployed bridge on a small held-out slice

from pathlib import Path
import json
import torch
import torch.nn as nn
import torch.nn.functional as F

if not RUN_FEATURE_ADAPTER_TRAINING:
    print(f"Cell 22.5: skipped because WORKFLOW_MODE={WORKFLOW_MODE!r}")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Cell 22.5 device:", device)

    adapter_out_dir = Path(
        globals().get(
            "MODEL_OUTPUT_DIR",
            "/content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge"
        )
    )
    deploy_dir = adapter_out_dir / "deploy"

    manifest_path = deploy_dir / "current_bridge_manifest.json"
    ckpt_path = deploy_dir / "current_bridge.pt"

    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing deploy manifest: {manifest_path}")
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing deploy checkpoint: {ckpt_path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    ckpt = torch.load(ckpt_path, map_location="cpu")

    feature_cache_path = ckpt.get("feature_cache_path", None)
    if feature_cache_path is None:
        raise ValueError("Deploy checkpoint does not contain feature_cache_path")
    feature_cache_path = Path(feature_cache_path)

    if not feature_cache_path.exists():
        raise FileNotFoundError(f"Missing feature cache referenced by checkpoint: {feature_cache_path}")

    payload = torch.load(feature_cache_path, map_location="cpu")
    X = payload["X"].float().contiguous()
    Y = payload["Y"].float().contiguous()
    sample_ids = payload.get("sample_ids", [f"sample_{i}" for i in range(len(X))])

    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError(f"Expected 2D X/Y tensors, got X{tuple(X.shape)} Y{tuple(Y.shape)}")
    if X.shape[0] != Y.shape[0]:
        raise ValueError(f"Mismatched sample count: X{tuple(X.shape)} Y{tuple(Y.shape)}")

    input_dim = int(ckpt["input_dim"])
    target_dim = int(ckpt["target_dim"])
    hidden_dim = int(ckpt["hidden_dim"])
    dropout = float(ckpt.get("dropout", 0.10))
    model_type = manifest["selected_model_type"]

    original_dim = target_dim
    text_dim = input_dim - original_dim

    SMOKE_SEED = int(globals().get("ADAPTER_SEED", 42))
    SMOKE_VAL_FRACTION = float(globals().get("ADAPTER_VAL_FRACTION", 0.10))
    SMOKE_N = int(globals().get("BRIDGE_SMOKE_TEST_SAMPLES", 12))

    torch.manual_seed(SMOKE_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SMOKE_SEED)

    g = torch.Generator().manual_seed(SMOKE_SEED)
    perm = torch.randperm(X.shape[0], generator=g)

    val_size = max(1, int(round(X.shape[0] * SMOKE_VAL_FRACTION)))
    val_size = min(val_size, X.shape[0] - 1)
    val_idx = perm[-val_size:]

    smoke_n = min(SMOKE_N, len(val_idx))
    smoke_idx = val_idx[:smoke_n]

    X_smoke = X[smoke_idx]
    Y_smoke = F.normalize(Y[smoke_idx], dim=-1)
    smoke_ids = [sample_ids[int(i)] for i in smoke_idx.tolist()]

    class ResidualFeatureBridge(nn.Module):
        def __init__(self, original_dim, text_dim, hidden_dim, dropout=0.1):
            super().__init__()
            in_dim = original_dim + text_dim
            self.backbone = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.delta_head = nn.Linear(hidden_dim, original_dim)
            self.gate_head = nn.Linear(hidden_dim, 1)

        def forward(self, x):
            original = x[:, :original_dim]
            h = self.backbone(x)
            delta = self.delta_head(h)
            gate = torch.sigmoid(self.gate_head(h))
            pred = original + gate * delta
            pred = F.normalize(pred, dim=-1)
            return pred, gate, delta

    if model_type in ["residual_feature_bridge", "retrieval_aware_residual_feature_bridge"]:
        model = ResidualFeatureBridge(
            original_dim=original_dim,
            text_dim=text_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )
    else:
        raise ValueError(f"Unsupported deployed model_type for smoke test: {model_type}")

    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device)
    model.eval()

    with torch.inference_mode():
        X_smoke_dev = X_smoke.to(device, non_blocking=True)
        Y_smoke_dev = Y_smoke.to(device, non_blocking=True)

        pred, gate, delta = model(X_smoke_dev)
        pred = F.normalize(pred, dim=-1)

        baseline_pred = F.normalize(X_smoke_dev[:, :target_dim], dim=-1)

        pred_cos = (pred * Y_smoke_dev).sum(dim=-1).detach().cpu()
        base_cos = (baseline_pred * Y_smoke_dev).sum(dim=-1).detach().cpu()
        sim = pred @ Y_smoke_dev.T
        base_sim = baseline_pred @ Y_smoke_dev.T

        pred_top1 = sim.argmax(dim=1).detach().cpu()
        base_top1 = base_sim.argmax(dim=1).detach().cpu()

        gate_mean = gate.mean().item()
        delta_norm_mean = delta.norm(dim=-1).mean().item()

    rows = []
    for i in range(smoke_n):
        rows.append({
            "sample_id": smoke_ids[i],
            "bridge_cosine": float(pred_cos[i]),
            "baseline_cosine": float(base_cos[i]),
            "cosine_delta": float(pred_cos[i] - base_cos[i]),
            "bridge_top1_correct": bool(int(pred_top1[i]) == i),
            "baseline_top1_correct": bool(int(base_top1[i]) == i),
        })

    summary = {
        "model_type": model_type,
        "deploy_checkpoint_path": str(ckpt_path),
        "feature_cache_path": str(feature_cache_path),
        "smoke_seed": SMOKE_SEED,
        "smoke_val_fraction": SMOKE_VAL_FRACTION,
        "smoke_sample_count": smoke_n,
        "gate_mean": float(gate_mean),
        "delta_norm_mean": float(delta_norm_mean),
        "bridge_cosine_mean": float(sum(r["bridge_cosine"] for r in rows) / len(rows)),
        "baseline_cosine_mean": float(sum(r["baseline_cosine"] for r in rows) / len(rows)),
        "bridge_top1_hits": int(sum(r["bridge_top1_correct"] for r in rows)),
        "baseline_top1_hits": int(sum(r["baseline_top1_correct"] for r in rows)),
        "rows": rows,
    }

    smoke_out_path = deploy_dir / "current_bridge_smoke_test.json"
    smoke_out_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("Cell 22.5 smoke test")
    print(f"model_type: {summary['model_type']}")
    print(f"smoke_sample_count: {summary['smoke_sample_count']}")
    print(f"bridge_cosine_mean: {summary['bridge_cosine_mean']:.6f}")
    print(f"baseline_cosine_mean: {summary['baseline_cosine_mean']:.6f}")
    print(f"bridge_top1_hits: {summary['bridge_top1_hits']}/{summary['smoke_sample_count']}")
    print(f"baseline_top1_hits: {summary['baseline_top1_hits']}/{summary['smoke_sample_count']}")
    print(f"gate_mean: {summary['gate_mean']:.6f}")
    print(f"delta_norm_mean: {summary['delta_norm_mean']:.6f}")
    print(f"smoke_json: {smoke_out_path}")

    print("\nPer-sample smoke rows")
    for row in rows:
        print(
            f"{row['sample_id']} | "
            f"bridge_cos={row['bridge_cosine']:.4f} | "
            f"base_cos={row['baseline_cosine']:.4f} | "
            f"delta={row['cosine_delta']:+.4f} | "
            f"bridge_top1={row['bridge_top1_correct']} | "
            f"base_top1={row['baseline_top1_correct']}"
        )

Cell 22.5 device: cuda
Cell 22.5 smoke test
model_type: retrieval_aware_residual_feature_bridge
smoke_sample_count: 12
bridge_cosine_mean: 0.854757
baseline_cosine_mean: 0.917815
bridge_top1_hits: 12/12
baseline_top1_hits: 11/12
gate_mean: 0.505774
delta_norm_mean: 0.643634
smoke_json: /content/drive/MyDrive/trellis_project/models/p2p_trellis_feature_bridge/deploy/current_bridge_smoke_test.json

Per-sample smoke rows
ds_000514 | bridge_cos=0.7894 | base_cos=0.8808 | delta=-0.0914 | bridge_top1=True | base_top1=True
ds_000876 | bridge_cos=0.9163 | base_cos=0.9833 | delta=-0.0670 | bridge_top1=True | base_top1=True
ds_000191 | bridge_cos=0.8455 | base_cos=0.8855 | delta=-0.0400 | bridge_top1=True | base_top1=True
ds_000095 | bridge_cos=0.6931 | base_cos=0.7858 | delta=-0.0926 | bridge_top1=True | base_top1=False
ds_000408 | bridge_cos=0.9502 | base_cos=0.9677 | delta=-0.0174 | bridge_top1=True | base_top1=True
ds_000898 | bridge_cos=0.9224 | base_cos=0.9606 | delta=-0.0383 | bridge_top1=